# Kaggriculture: 7-Turn Rescue

The 2800+ in the notebook title records an earlier agent’s historical result. This version repairs an opening liquidity failure in the previous Rescue controller. Its first market action buys eight wheat and sells three, retaining the planned five wheat and one planting seed. The smaller round trip protects cash needed for early worker hires and feeding; a demonstrated failure previously led to a cow escaping and lower production. The full production controller, input limits and market-order handling are preserved. The final seven turns retain the bounded harvest-and-delivery planner with 128 simulations, one pass and eight proposals per worker.


## Local evaluation — September 22, 2026

| Responding opponent | Wins | Losses | Ties | Mean money margin |
|---|---:|---:|---:|---:|
| Previous Rescue (v2) | 14 | 2 | 0 | +8.0 |
| More Wheat v7 | 15 | 1 | 0 | +98.9 |
| Master Hybrid v4 | 9 | 7 | 0 | -84.8 |
| Evgen Dvorkin v23 | 13 | 3 | 0 | +664.7 |
| Fieldcraft v4 | 14 | 2 | 0 | +2688.1 |
| Hacked Stores v3 | 14 | 2 | 0 | +1788.4 |
| Synthetic opening 8/8/5 | 15 | 1 | 0 | +171.2 |
| Synthetic opening 43/20/18 | 15 | 1 | 0 | +189.6 |

Eight previously unused worlds were played in both seats against frozen, reacting policies. This confirmation used 240 complete games: 128 for the new agent and 112 matched previous-controller games. Previous versions, four public implementations and two synthetic opening controls are shown separately by name. Synthetic controls keep Order Book’s reacting policy after a same-stock initial trade; they are stress tests, not additional public authors. The same fixed weights and acceptance criteria were used throughout confirmation. This was the second candidate confirmation: an earlier full-controller replacement improved money margins but failed the win-point criterion and was rejected. Its worlds were not reused here, and the opening repair’s already-tested source was unchanged. Exact actions, execution ledgers, runtime and source versions were checked. The refreshed survey found 18 distinct policies, but many share production ancestry. Matched seeds fix the initial world; different actions can affect later random draws. Small world counts, related opponents and multiple candidate selection limit these results; they do not promise a leaderboard rating.

Run all cells to build `submission.tar.gz` and reproduce one complete confirmation game against the previous Rescue controller. Agent and engine bytes are pinned; code is collapsed by default.


Against the four ordinary public implementations, wins were unchanged at 50 of 64 for both versions. The win-point gain came from the synthetic 8/8/5 opening: the previous controller lost all 16 games, while the repair won 15. This supports a targeted opening-robustness repair, not broad strategic superiority. The 14 wins and two losses against the previous Rescue have a mean margin of only eight; many are close matches between related sources. A known tradeoff remains: ten matched stress cases gained a late goose escape at step 647, although the early cow-loss cascade was prevented. No previously non-losing matched case became a loss. The worst paired money regression was 2332 in an already-losing stress game. The livestock events are included in the complete resource ledgers; the late goose behavior is not claimed to be fixed.


## Credits

The controller builds on [shiiin9’s Order Book](https://www.kaggle.com/code/shiiin9/your-market-list-is-an-order-book), Ahmed Berat Özer’s V55 and [V56 input rules](https://www.kaggle.com/code/ahmedberatozer/kaggriculture-v56-smarter-seeds-and-fertilizer), and [Thomas Tschinkel’s Metav4](https://www.kaggle.com/code/thomastschinkel/the-metav4-farm-submission-v13). The seven-turn planner and integration are by Dmitrii Gluzdov; earlier ShopRouter work is credited to Yusuke Hayashi. This version preserves the complete previous Seven-Turn Rescue controller and adds an opening cash repair with observable checks. The fresh [Master Hybrid](https://www.kaggle.com/code/haideptry/the-2965-master-hybrid-engine) was evaluated as a competing base, but its full-controller replacement failed confirmation and was not adopted. Original notices and Apache-2.0 licensing are retained in the archive.


In [1]:
import base64, gzip, json
EVALUATION_MANIFEST = {'candidate_source_sha256': '4889137f1adf1f9266b2ad5a6ac6700c40e9be6dfe3654e39db375fb55b71341', 'candidate_entrypoint': 'opening_liquidity_agent', 'opponent_source_sha256': '0565e742904024379315cfc4d3228616a214c7939d0c7804f494bc4e3d4700c7', 'opponent_entrypoint': 'rescue_agent', 'engine_distribution': '1.32.7', 'demo_seed': 861795495, 'demo_candidate_seat': 0, 'demo_rewards': [87467.0, 87459.0], 'demo_action_hash_sequence_sha256': ['e95796bd0054b6bd83405e1ac7c7851c57bd32ca971e6d60f47126d1fac9bcd8', '30d7dc0e8aacad13c6ab65510539d13e0183066c0988523dc5083b4599ffc44e']}
LOCAL_EVALUATION = json.loads(gzip.decompress(base64.b64decode('H4sIAAAAAAACCu19W2+cR5LlXxH0tIMlpYjIu553B/MywGC6gQW20TAiMyOtGlOklqTU42n0f98TJbvNkoutj/pE+2HX7raoYt3yZFzOyYyM/OvLqff28s1LIcmX1C5FXl68HDfX63D7Tu8PN9ff/eXm9mrevXzzp5q5tBRbumiVa5TQyoVQpBi4hnaBXzJLSSlepEZZGn684JolSg61XjAeKxRraxeR8Dilmv7sn/X25vYe7//Xl/92ax8PNx/uXvy73Y0P9uK/fZR/8se/13eGJ3C+ePmXw7X/FC9eXt3c3fmjcvHy/uA/0MXLd6bX373T2+8P1y/f1Fd4BF/+7v7vD13GInj4bxcv//Xm1l78r7em9y8+lvOfkX75DH7sM1p9VUv6Bx+jd/d2++Jffuy3h/niYzz7Se2XDyqPfdBlja/OfFBqHI8f9D8/fm/XL/7Hx5vbHw7XLz5KOD+m8MtHhcc+Kuf4Kp8dVQrh+GH/fLCrOW513T82pE0TJLkCpSy//qQSWzl+0r/o+MHmiz/cY7ruXnwMX/9hXPBhMZwZVhD59GF/+PH6/q3dH8aLm/d2fbj+/kV9XV+nr7cOLvyKzyKZa3rkI2N4LfSa645Pre1VOodqzDnjU/Gxdzcfboe/9k9/fXlrC+5/9/ZwOFy31z/iN5d4xQ92f3l1uLu/PNxd6vXlze2028t+c/MDooN+uIfL/vIiPPTRbu8QK474X9/cmz/zu7u3KinjeX3MkCjMJp0x8l57X2RKccUYKISYWxjTCmfVxYm7P3HmOXgFBVb4gL+/l3I2stYjDQ8lBbGFlHqbeLeAXwwe2ilwaktLzSXPXqKUMmucVbTjvXxc313BM7+7/XD93f3h3Un84z9KAORvpL1qNfxvPP/W7m8P9tHmd3p/Ein/yIQ5BuCvJCLitf9O9IYIr1iw1rffxfr25Zv72w+GB/Td4epHvPgPb2/e//vNBwSFy3+1e/0YL//t8N58gB/635/07gZoX784XL+128M97P/WX/ECUfn+9ubqym5f/u3i54nTt+9sdrvV+5v/stvXP+j3398exoer+w+3dvkx5cs7zKZ/3J3Z9Lmcl8tu7w9XBzz94WSevtHDOeVzczpNAuw7DK3Vei+VUqZBHCWFGPvQVupqgecKsRcbmPAmmJGeVNsYdjqnOmlNnlxiLMVMRp8zZdbK0kcvJC3HMizE1JqVNnMvzRqlYmvlsWFOOb2RCv95ha+8aU7bKySrGun3mNNrvX+r1/+h46Z/NqHv8cZcL+8O/3l5pT9igh7O4IOXfXH6bKWUqrJ1GqECegRBGmWpEZXQZslJapJg+GEZpb4GprFKqiEXTuFk+qKmHEbOpjWWmMfsqYMSRLZSZyu5lK5J67CVizXlVHRNePrMQmWWLS7Jb4K8CfQqVdoyfUKvQigtl99l+g5vDz6gw7i2D7d6dW33n03juLq5tstbHXaJqTv8F6bp4Tz++vUP5zOdm884rY8yBHMxciTqPSI8wp10hIGZTWnMODnAuRBqA8JixtxSbVF6XnIyn7AAWs1YQONk5katJR29d5qLLayYuSZMOZXCAz4SJhUEX1hFTKOUumU+j7OUyitM0ab55FeJCFzy95jP7z/cfnh/q3fq/nXHzJ/N5rsjybu0a6RZxNzwcC5/9dqTZElnXbOSUc7LJtWU8qzJtAP3GkyHgMakgBRY2BpTFLhwTJhBUCarTCu1h1MZAnxcqS7NmE68EzGMLoXVJ54eprRYEhmHmRdIPTy/IwYLNcXEUKVtkTUFd82wLbKKYNIjSfo9pvLqx+vrO/0BXnV4vfT2Hd74P27eXh/TIojY5WFeGTLl8VUPZvHBy75IdvK0VPosEm00qaF3RMLMrfSFjCWj1T4j1a5tJZ0jaww2IscqdcQ1TjyRR+cQrIVoYYIJcUndJGeZNnhSq9R5cOgaQo9Bmy0KCWwo+CuyrC9On/yRwhvCDCb41zZPjK9c7+X4e0zfurL/vLnWtT4PqO/w5+Hy+MpLn1cw6UuFIrp/OIu/vPrhJEKZnpnFGtccY8XqFANRMCYkr+rkIwmHWooGTMVQ/K7XKk1GgPYl7ZMjvDCezCK8EoI59cY54JkaWm5l1IB0uGJKc1XWEufq2TqrjpnBhirPRAMMd5MT8hviN0yvYtvmhOlVzi3l38UJQVIO6/rGbm9v1uH/vNbLv9xc49XrwxUkx7KHU3b61C+ymhSXIhdNRfaDa2X4FUcEs5h0BVtwR4iDpJgs7TksAbMhf/KEu7ZkJ6QUcwtSP0OVAhY7wEkNUwiaBOlR5rTlWRVTVxfyYLCO0Fo6MBXJM1ieX5w1+iPJGylvYt0qNCTD9xA7zwqNpVd3T562mysA/+Jfj3oPLxg//GreXvzlcP/2Z3X6+i/qyxm39l4P4J6/TKn58sP8tPpw6poPZ/Phs07iaDg3mcytUc/sC03SQVYjAe051shFKpEmpx0tZ82j8kJSmzMOauCqmNp86oJtrMSQnNIL4nFm2AYCQVpQJjJSEyEtkJb4lGEgNSMP6iBTQXSpkm5xwfomNAzllaRtFLW8YoTp8ru44L3d3324h3O9nvbOk997fLbdfvS4iVm7vrxTJEIMEg88nMG/v+7h9OVzs0cZMk1XL67q6+CuyHqDiGwgelb45QS1XDMaNEAl4B0hErIIKA8kw3g4e22Q1iYIjbGtDjpDM4VpTmXbsgqFr9Wkk9QKVqQTqZIa6YJbtsF1i8DgCD98k1zJb3PF+qpSC6X8PgH0Zt58oCafZcG3evvR7u4vr2x+f0pg/v6KEyFxduKqJoSxqR74SAlEX5GNusaVrWJS16iqCHqty8gyJnFlsP8yLaSQIp+4HSgr3qmDsjbkORulAzNIQQoIpqCsLMRBYp4R2lII6bYi6oZcF37T2xb+ArcL7kxlm5IAUXUzod9FSby//Xh3+FE/C5GX6xZPPuBTnILe3RyuLm/tnb3rmCiIxMOJOPz5LU4m8qyOGL0C7UQzhs6ceUhIDJ+YZaWSWAwkNCDvSYA31cBwu5kToiN1w+SWhxMJptAjI59aDKUXRM7BTYOAkMpqHQQH6r+sHqA8Be82WlFLK/cF/5ZctkwkvYlHekmyKX4GflUjQRX+LjrCbt7f3iCT3XwEf7m7Qdi8WZcH6PnjAtvBJ/Xwn4YfwTkP/eokDT548QmjOZsFDaKeDOyzQddPTasLTQ9zsogikiBiajWksKIK0bh01TS5trrGWqOdZEFPmWIK+cAUSpAKVRKGYaoQnRNspWsJBXokDeTWkfEJw2JTZFMyo7AljrY3FF3Yp7TNHaEGEwzkm8XRb8Vo3uph2vv72x9fu1NKy+lnnf/2uJnzk9w/DbI/veThrMazC+KcodfigHc6k8nIedJnb5RtzNCTBKrVwE4RO8Ft+mJkUqoZAnDUFk94anPfRZYDWRkInSxaGHK/cGFBZK08Evju0mnQ3C165J11qRXMPdx6k0aMLvFTe8XbNGIIr3KLXH4ndXE6cYku35v+cJSFX5ytsy7YIlQeWCbIBwSbJJDRjoDXoNSagpVA0AOYEgtbBh1tM4FXzlJ7D2lC8D+cLSTVOZdA1i2NUQwKXtXX7uIYjeG+Zq7t1woh4A8tsxYwm8UzRmhI3uCCUPTwQmmvwjYiGlzRU/t91tb6h7sf9f0tBvG6yeXHu2MK7E5i3n/oV4dxnLZLPe5egNdgHg/Xlx9TfDiTv7zHFwWiRGhrSIqwei1xIHjWskKrFVGuw2BjogzXGDoowBcRxEBfB9JasgJpnk9zoqwQdWIGQV4iJlvjUJaZ4YirZieirixXgXyEQzvbLWXEwJhk3iAQP7FS8v0lTNCmqcyvOMZQfyfH+2Hiu13dvH9n158v0IBzlcu7cXNMjD/vHv96ieb0Pb4YR8ts1j3oDUO2yphMIqcmCJ6IfEiMHHUA7gqOCcc1zAx8L0WIxZBAZR9OJxwSCmLUimmyFiDh4Z7QD+ZePBPMokEIUgkLeZVWnkVzBYH1DZGceNN0+jrpGyqvpJWN01ljFvpmG4t3SHO3epzAd+913D+SGxec7YXh0Zt3h/EwKY7bH+/u9apr//H19zfz8u7SDQS6w3fpL++Ou/QPp/PB878cZaH9EAMVKgJemqajqmCaUPbRmKZ0Bd+LI7JT0YZkSKn3jAzZSpmJHs7laqnWERGKBcYgDdYAeoPgqr14Wq1VC6Zw4IUU4b0wFVBZxEskWwFx3RZlQ3aik+u2uXRKBF4Wvi3R+edPi51/cM96OI/vrz7cvbjD+1y+v3HWcri5ff3xcOdU9NIffnGD2bjSH1/+7c8XP9cZ/OmvL2/ev7+5ds97c74YB18BtvHyzS+lQHjABwwFMkB9D15N9F3X6x/wnBJz8Rqcn9/0l8dBJbxq4WGhzke9OsyfkYBpeuXRW717e/xiL63BXXOfRCl2/FkDqL+xjjJKTaA4pc8gvumMHOzLp0jGkicS6Gh9TN+SCjTLHGRQr0Mnh5G1g3gxQZvirwYdGyjnQcgGIGATFLiG7hvO8PcIzvVn94OvBoifF6Ddo7vYj/HnAJ3WWH0b0wFzeogM59/GeLQhB2gbHYm9IX4wNG5luPNErJqIGZMRnozwTwi+T75ydk3liakS69diw8+Nze6BPYfdfF409yTTAU3ns/AgETyE5zJ/et6X/CpEEIw5oZommRrZIud5veOHoiB0GA/UNP4rA7xfcihQ6BkaHczDlzGT75yvFUcGuVgdEqzSqtDYOU0o9MW+A0Qz9mHQeQCdVwF5ROJvcaxQduDDz47P7sFd7If4c3x+XQv5tNgjRc4B5DvoJ/7FsXwZoAYmQHN1uFmmkWPtJUYec3QDOke3GgQSYpAZ4JJi8KJSnWlaHXwUOjVDMYaOP0ShQLtvVYkIWBr8CDG+TVsTbxmB1RgCvBhfdvZekyMe9gDEzw7Q7tFd7Mf4c4BOC1yfZj255XPgQL3zQ3AgiNKXwYG7gBWTy5beWhpasyYID0Tn0WInxXgaH90qrAav4Y7sAPfCmBVcwMFBju8uTHkMGnjSEtDhACba4UVVfBEDLjeyc+3o1ZmI1Jp6Ut+7KfNrweFnB2f3yC724/s5OL8qWH6a8VSpZz1Lwkls5vQJxy/gYzKmqnpY5pyTb1iRuO6NAEhK9nXXAaGS2rI2Q2oI2FHxr+/GLV+3gkKKKyvnday1rUI5Io7H3CGgw1AEYw4N7zV7bQVSmY2bMRgm4OpsO/DhZ8dn9+Au9kP8OT6P1aA/MQbxI7qCTmAS3uBlCLIjNvBEZHqAYR2Zt2qvXYZ5oe/0hc+BsJogLyYXSmMsIKZSqpXmpaicAHRMneMExTQv+VZApcP/N6wKXLWHXGMg4KJBqgFLMMkItV7mN0CJnxul3UO82A/0l1H6++GCJ5rTea8roZ3ke/HjIV8CCnG5uLdoV+uVMNRoBdSvOjqtLq4JHM93mduMCMEdeEUmiI1YrWWfYx2gllMMST71pjMqAnpDSE9Dkte/hejVdbNONkmpllVzzcgRU6T1TN8EKH5uoHaP8mI/1k9a8vjlUNjjtsQhSAlnMPJf5PTERY+ZC2XL1Eqi5BtcmkADE2gkPC9jtsRWLRC1pSPNy6gry+gKuuDl79GLIazDNzHUkQNExaLAi0bkglC+pKhKajU1aQEsw0JPMUiMA9yRilcE2k6I+Lkh2j2+i/0ob1v72G8+4YQC0G9kPy0jvE/E/bF4NS3I8dMUQgRsc4JbaJ9QJNAZLlEsfCpX05AKFMjMXwsOPzs4u0f2HJbz2MrHNuP5dODwHD4QICdrH6VtiNA2QB/LCDMSWGNL4Hgxy1qljliC9li6jaVlSSQaIS6C0KkzTRWFRnXuS8QtINNrTQXsErokgTbG0UuqE+QyeDFrWdGOB6nUYkHSbykg+jNPqTsQ4t8Aod3Du9gP8vbFj40RKLCch+in5Y5f1l+38EYvlUpds7UlWfwoBlmGrrXlx6aHgQDSmG1CvS7fTUZKb1AnYBTTC5vx5TMyOV4CWdsARh2Aoo4eGapmZV+bFeAZe6ROOQU8Ffk/l2wGj56AbQ9C/PwI7R7exX6Qty2BbLOfJPKIi8G4T9D5KZr/Y3gQWvvoHjvbgGy1Bu9aFPGDbzeCEJbsi/ZpEM9S1sTo8LM0mhb9LJBvRHIu0nWW0QRhmRLCN/hf0p5k+bG9OMnP9FHu5PtMllrJtIafr02hz6+Fh38DeHaP7WI/wpvXQLYZUKRPbvMrhLi1eiIzKG9Yfx28iq9uLRBAXal4GZJAd9Scc+XpZ6ob6GBcCgjYiljKFaozdEljLSdBhGdmV6nkHSAClTY7ES3NdYovx+KbSNThy7cj5mzJTwbbskTqRfc7EOLfAKHdw7vYD/JTV0A2s2l6hDCe7kAz/yZseunEv3GtSCnDpTT41sBsiNXLg3ocYqCZYACLbZSURpwd4iW1iDfgtB8mfnaYdo/xGXj1hgWQrQZV+TGkThfV2obIRLmJVzfXEZoOgTb1igW2SWFlpIoYaUVE56w68P+2lL1sq+AVxNlrJc38AEO3hpBekwqgTKABPfbJVltaIKIjGt6TeNDCJ6l5v4FsFfr4jLz/GqT42ZHaPcyL/WA/aSHkQdugf2BOQuHsfpH/otanLhb5LnOGX4TYKzf2liMBX6Y0SyX7GXlf8urauEJbpAYXhF8tP+TCJuTr2Mf97UZFgxBQzXP5OfoeCty4+yocnh84pdTBMn33c5SJLBFH01Q56F6M+Lkx2j3Ai/0wb1sQ+QYGlJ+8XLTfglx3JHzzAiph0K1xld6qi96uKZQ+WKBbWlFfqF0xCoGlJshbeG6rXb8aHX52dHYP7Tls57HVkK3mg4R6HqCc66kY2ZLOWp6KeAvJAN/JNdUUkOc7YihJgALViVArXt9aWhTkEJBNao3U+joeKe+mutpK4h0hamnDNx2R0gdeB32UO4tqrLGDQa40QB28OJ0tI2LXfGYv/wkA8bMDtHt0F/sx3r4WstGEmPIjCNFn1TIUNu3GrlDL6IjDsUSe0A7FoFyJBo8aJ0Qur+TnisPMg0pdYbD2mRxMnsd6PcCyRgoj+1Ha4cVUk0vVphUJnQaYJRSyBSauvfh+dshaSpDBQF55F0b8G2C0e4AX+2HetiSy0Ya8TdJZfBKXz+J04C31sJpnW8jc1ZC865g1xQBfIcAw/PhKtywJiT76LhBFQBCoxTkCBnc8uIJ3gHBhIEPSpKyQk7uWAqdaoFu0+CpsgBKGjjEFwDOl2afL5Jy4fTU+/Bvgs3twF/sh3rwisjWTSebzLsZyWjoTNugNQQ5OeOnsHXJyqPiwefp6bNDWpwoytp+NQ1DNVbSoQoeUUqA9qEyvigUn8HW0APhWieCTtY9QQlHxNlMakmXtMhXxWnxNBY7XB+I3QPaCrbIHIX5+hHYP72I/yE9dDNnOqeUR1tjoRLn+JpRaOo+VvAvZmkO8oUQI01f0QUO96Kq0uSqYJsiF9zLo8GZoZETuESpeqd8AJn5umHaP8Rm49YaVkK0GFSk8JkM+q8TaUJDeEKqTIU3nOszBarPBw5Dqo7e0UC7wRnBJNeLYU5bcpWGUmYJH6CNB6sP7ONFoAk1TahYV30DIwVtUMPI/Iv3CZPCg1GfrhvcBfJijEdM3goqfHard47zYj/aTVkQedJh+3J5S/akY9DOM8Lg3l3jSeojagrho5l0QVayuTr5Mu+A1WcAP8pr41Yh+tkXLnMQI7iVCsyDNR3YiI3h5XkBE/dThQN6PSYQ9zXdQyFmtVqiWsNjj/ezAqhc/5zDmkt7LXoT4eRHaPbyL/SBvWxPZaTyU49MPV+03H+6Jg8LNonfAXYjTcM5yXNstS/PyBlU8wqiSSGKZidcYfs65wg1LDV8NDj83OLtH9hyW89hyyEbj4XQWH7zupPwqtQ0MEm4yNKzR2UYUSFDxo1UhJm8Z2EoG/5sgjBCecDvXqjWAFtbeheNMzes/MWFebqNmdQ6XG17jCUqZ4sCMeZRmEtBO07hK6tS0Z05zclOKQ3gPPvzc+Owe3MV+iLevhGyNPmeX05KfUH8IUKRNBXxjcQ9QAkMmhCcV0twThTRBESEioDpnHCMXiM/YV0tFDOKjtlDTbF7A59s9PKBMrfJihjrpA64FQqDJmw0Z9G0g9VY2YActT2SQaWU1Uf+EuAsgfm6Ado/uYj/G29ZBtllPaeVc6UwCrzxZRQs5byosinkEWZG9iM+lgjY/lpe9pXBX75LSEgTFpEBVJ9wH3uObRiElisdVkJjGUCp+DiIFhuOEYwBu0xdLUqYEJVw0zSIaW0H0LwLWEOqEBIHD2lejw8+Ozu6hXewHePMKyEbzifmctki5nlbtZd5wtqr0nJHUc53srYwgV0fIUtlowidotT5H1UaDS1Ak8CXmbarYHWUiizs+Ikq0CvglZ6KMDD+XRSCJH2JccNcFHhkMEZogqkEdDHRiLe+dnULagw8/Oz57B3exH+KnLnxszGFBzsKEdHy67CEb3MwlJPyF8b+0JITjEsUoZSUG0yNWMD8ozeX3EqyQWpBk1a+FQEymEnyKeZSoWhOXyeAFAKvUGWwtLiACJQfE6zmS+i5QjFQMIVycZ3rHGQT1bwATPzdMu8d4sR/pr1j02GpQIZ5XHe2ENYLxb4jbsc5oyRCStbZl8IswvJu+X3AA8PxIebc6BjRMRhyvA5E9Iblbi2CAnnSidAT6EdaqLIIHTZvXI0uPPVZbVRpYpBCeMeDl4AGLvEPGMMFbcPs2SPGzI7V3mBf7wX7S2scvl449bkxZwtmyKzxewhOXPmIETyxFec1sfiBP2Zd24TRhLHhhQOAlXz+rhYLfWUQVoB1PCc82jwBlcEdV5tA6RAi+Ayt1wCxrWIG47RmDmi15B4011CIwI6SJXqFfZc2dAPEzAwRNtvyKEUIiigg/K4lf2AS5NqWNsapvarQAnY8EZaW3VquR91Mbfp72uM8IMmUIQb4KTWllRJpmKcISreZQ1I80ZpqlBuTBARqu6jy7BGOw8q1nzzaazlkC6S2rT1c/Pi2iPbvxxGQDUQHBnMEEIEO4+F5QXh0qaWWWBt0LjYP4nVqyBd9anSTxGKO0yl+LDT83NnQ8Jd+ogcPMGoo3T1xISm3kNmZvxXzHDAYD9e7Uuk9EGRBy+JJpkfEsdvPYyscm08Gw01l42mmt5yUmvWxZV7TWhrS+QrZFw+9uQSDGaCBcZ4aq0MJ+p5b5YaoZ8COytDdX98uZjkdiauS+ZmogkFJrTfBMoKUQNHEgCk1om77UHObQCYwy4s8aj0G7rNB3AMTPD1DByCVXSQEGmCsmNXnba5qMdGat+ElqLb1HCd1dzLhl37iANM+5H+9gwd9BxIW932v2m1emFKHEvSTgnXwrjme3BnItHCj1sXxLDSl4jAaTfcLSx7boc75tky98ncTlyyhbVj7W9EtcxMQPuXDiqMfrnI41UU1HrAlAgRH6hSXwlRwdAIFLtsLHMx/Nr1poAxkq9r7YIsJQ0NxqNPKV17gC+CQPvH51hHy8WxU/hoW/kVnegw8/Nz5LfeUQ0SCDcSXE0Yi/LfgA/G4iBSNBQbwHBFQFFcri9/jMEhDUoeXi8VBVh/ACRDX3hiyPf5IvSQPjLhbrEF//kenvBnUW/DYt1g6nNa1lnqvRP7/2sc12ytmGTSmFeOJdXoOy4UBeKjw1BSqKpO4FnN5ZqVQXDgXDSguZHa4B7hcndAlG6QcbYoCNzHy8U4oa8lwMGTQPwT2MWKj6PSvIhR1vPUkrhK1f8zG9x0+AJ7rGbciFmaTErwWHnx2c3SODZ621BNEdWbsCQa3HXjFxNXVwvYkjjDNRXMC/MuI4mIQTqZpArkN9Qu3QJuMJcrZIOBO+yomoKG0DPtUPSZE30tDREGAaI2g2DHxSRkbOxaAGEGaqjgx/m16UGchvgJxw5OOlLqGxH74SKI0F7hxbqLKABNixp+9hfqwTAmWGsNK01L3LOTneiEXzjGdtx4efHZ/dgwMvRFCffl+K+FHcPI6iqhCMzfdj8SYyC3H0tjSwlxEBLRJbs8wG9ljDkxc8NuavKud1xWm3Ha60pXOTGvs5O782xFe1kKaC3ylpNrztDvslluDNK0iINCqkrCL6QnJQBR32EAQSSdkaVOsI1SuqvKIBb9Kir8Bm1RmR6Qyxmf0VXpuHjCnFvNtwpbIfJX5ulKqbILdOqXYC7zU/u0ogcb3MUP36JOSstrT5VY46Jmi1gkJKmILsf+zgAE1B3iEvIaObWIfegJQo0G8ztgVVXxMiO9hlTADLOz4KA9cABY8kwPVr1jq20qFwXnC0U8GxZSsRAYcdhuLXCLPB6zI0OcjgQMoiARU2v/PNoVu1FhgSVIbfrFKgcPW4lVi8/GpBm2fwBdduM44Jh5rg4qPl6Rc6jtZrUA556cJ09Fr84sZWmmn+JkDxcwPVCPEHDNwvMFhRlsUwMCRXV4jmEKeqLAOmBKYXpmbYUyeFNUDggj72Y/Mm8Q6g8KfIiGplMX5ax+Y0bRloElcQ8x7BJQrCYEEGLV7mFmG7Xu/4xHqYX66ff9yYQHvDuV3FwjnlUwqwgR3NAPeSiECSFakevJkGwUMI8R+2MRnBKYWSoTtAA6D1m18Nif/4raqfGjHnNDsdT1cBrQwcFstxdSQiP7L3nw8ZmqSM5R4NjWx5NE+Y+FjY316I+BxEOcezEJXT4tjLtKX0I8IpEDcQef2CC54RUXaB/xD3kb1dmt8SVQ2PWqo5+uqRb+zPNWP+tDAURwMLj7WPAfJjCP+wISh40IkFp3Q96UtkyIAQuMXguW5Kyy9+GEqRttbFbLQfyWfBSafHFzC+38SAWhdkKEQuuFEw/NAVdmM6e/Wr0f06mcpKiPIqUILSvNkK3i9UgsxZ/avBecRyzjtXaicdZfKWmo+58qIevUkZQCrqXT06jB4A+cGeySrglBgXVwQnsoTcVWYL+JJZuTyL4TxaE7PJdsL5fheFpZ1mfAxiw7Ii+HLuSGN+4bPMChUWY0jmPBJcR8zbd+MJHWQvCwQ5Yu9qQzSUBaLouQXsXCgsyCIonqRHr/Tah25OP30dcni3HljcxF8DeBWlAhlbyWKK5yrOtgN0zn7AutpZgELmU0okW0ryUvJbdjztQFYQUBm1eQMijNn34MEnYVsykXagr4pX5c4kzpV6H3zstwOiVFf2hntzKNS7l36s7vfDmMHrkMPwH4aCgQ0CGD+KTXMGRPIIanTmTPk/KIrZZkJ0tiimCKcThNqWo+TQHANGkuFWEPdTagZjNjV4yzAQw9Zjhv6EPp1OnqM0o1WO5+koUKxHAQKKA6ELRUfLq4KHn4Wo1a/sKJG82wriUSb1kzEC0uRZDnTLkAmYZR9AZ02I5KwJiYSTnfvStpxSnK7rQQEDYiiiNLhMgqI/BuOQIoK3H46Hg4HvhKXh0zoQ+eJPRGxxgGqCoUGNgChmkOpFDnlCCEZSBL9ec0IFIjwiRNNi3w1pnI73tIlfEVy31sVssh5JZ60HUpxPTnAiEW9Ap/lBlFEACEwAXsFgz2vSsXNKC97HUuAFqUnMDZLC68nBkPFkT3vj2HMYDyy8qpVFNiYoZwqrgRUQg3ZC/nI93o3m1/iYFu00wsQnLvaL6PuZneit6Jw1HaTEc+h4y+ETdHhLx2q/ix3KlVwSpLKsas8NZEdgRQaWAy3vd6DCWAhMr4irNYEGq0RjlHzsdQ4FPxF5IcFG90wIus3QubNrTh71Q44IQSDivtZkHfE5IHoDS7/NePYnVMVsMp9YzypW+FY9aY2CyLGhkCGsiIyU4LEDA2J4hNMHQ1D2nRrEGI6IE4avlXSJN8qBj9GKy5dr2/H0RirZ+1hkvzUtWfU84Rc74W04FEwAKCXD0VcLlmaKEHgDxHrONKe334l7AOKzAMWz5FDCadH0z9L/C/7VCnTlsoTgXqNZNvFlQW5+LtE3a1LKWY4XtI7kNxtNr3Atw9eUuJpLVdBJP+TTmXzF3tTvgPHhkzembqXkXL31LPUJg1KIU0SxkQJ4ekWiG0/vRbSRRNcazouwU4Xx0zG9LyzgFxhO95NSvhQNdlehVAviQ0SIzRwmHiKGNSF3TU5+Ilhrgy0nkL3EDtNCqvLzwGLgUTl6h8vVvbVBbRWJVWLvwe/486sZU/D6LHDSHAiEqXLf1DrmSzCdpdP1bNdqwFQ/O3Dfthy/87WhgHyO/I6I78tmDRF3xWJenTgZQ2Lkp4ZvKKBIw6DcfMlWfXfN3W22lfzSGCT/FbxWttRUgzehQd53FpECZtALqC15Uz2Y5DBblSl46dD8qmqYjQbVzkvWfFrwKWUDUsiGycbIyP4aEEfmPK4PItt3RHKvZxnxeGYc8braLITUtvzK6+klHfW48hEQgdxLVy1DkoE2UAsNsmR5F14/RN7Vd6j94IzAvxG2DUGhQt/Age3bIHXeph7RIPm06eem+zw4MhwCWpUg1pmtcyolIYvHBA/v4ntgyNMRpAxCKxQpCMRSFa6FFP7p3EJgJDbEm87MCGwQbIh1DL+DDEN4O64bJEQzAOy2dFQf3WnGjN4c+4kLIJQFRL+2f3xKseSzJQ7HX9BTb+vKvtHHrTdEbtBqWFRD/u4gycj83kIg+30VgcGMecLGevH6Tu/8jbxH8dP56e4EykCaNJJ4K/hevSuhawzkyJRSOV6E6HcXJIolQqsskHRwJXj1Xoz4uTHaPcCL/TBvXAfZa0CJ6cnHp76BCeVY/Ep2UG1wxTWzH4AAq55jxuqibkD7wcOSV6A3RpgD2RJtYbSlmix8NTz8/PDsHttzWM+jCyEbDajR+RPlEFunbS5ky1oRdVCfmf1EFMieX8k1vYGgHLufBqQyHsHPMfh5g7UQp0cqNJskRPe0jiXo1U/QtF6zZpBtmZCSfsX0XCv5eQjoDcgU8V1wb04gElcE9yozFSNtansg4t8Aot3ju9iP8hMWQ7aZUT2/oAaMyikxQrrfsOFhoDzeGufY4qS7NElift43+i50B5Us+XhNjrngyqUPoOFMikZO/XijNYFOQpyy+U2+M3BvFfQdpJNb6lMjoGM8I2byMlLQAuB+LFQVmpJn3IUR/wYY7R7gxX6YN66KbLKhID/dNfkrfCLF0ysqfioA/cKyNShzhYDwstQ5EhQXeDLifG8QWX7/SYU+Ee0lNajeZRF4Yqx+VS8k8HFVFr/to1eXsd6rACJOvOyzLbHRBoRJa34rz/H+JUho4N8nZiHZhBYulL4an7OEGhN3jlAnKfG0GI23wJNcfnlhtMjxjsVK+LcYbHQ0QiJaJr2sAKXq7TunHz+sSF4VxMhJsxeFcJze/aP7BgCzL9T3AA7tldkJRtMQe+oQ06gTKa3j0WKgXcPAtMeTWjRvsiDIzLMy1o+afHaQkzdojuR5XEa01ZdbfzIOEeILpCJlCLMKdZ4RiZOtYKphQaJ2J4ouNPR4Vw6iutbhz88Moe+NNiJAzd0YBjdHq/Aprit19RWSnEExcx1VSglTn3LT2UYTKimeX1fj0w3XuGXHtVRpMA8/A7+qraINugxSrLPXaSTknjnNu5cjbcEMqCw/xuiN02pHhtJPggMGBJIG4yHwHL+DG/YGxgTJ6znLG832KJBkJRY/2J+JxGtqj9eezaevh2xkRVXO923C5J6sQCbasL7mhXZuANOP5s0ekqnfPdEACICDOyyB/gcvnK5lyW95y5kkzQalOm0dK2P9PvHh/XgtB6hbRP06i9TQ3KQQvWfm4xGPTuYXlPOQ/Gl9hTFH+g2A4ucHavcoL/Zj/TULIluNKshjWJ24X5INhVi+v+jtu0cnb5Lqe/oI0dSTn6UbiOlTVmuh1RYVISv4RacL3GAuyn6R3LGRNWIMfkdRogEJK8PPjlJc1EAjY4ZXk2SF1GnEa3nfeLDXWUE4GdTj22DFz4/V7oFe7If7SUsjyFA1U6r/qFyt+SGEcxgR0vfpvTqxbNnG7skQz0ctat4TRUJsqyUCHaC8ohcfi6pvzcNtpMeqoAMNg/Uz6l5Y4tptqsXlx2ccjND9mGgdNIiQ1Xzp128kAHrsstlPFSmIUk4xAl0/SrwTpXOm5Gico5WAr5ywSm+msGUzya9UQDgB1Z4YavbNQ2t+dxBZiFIbqdcVd4wrgz/71llOLc/UfLn6eMqhBFXf8/Z6m1I6Ve+y7nUQywjkEbIvVFoGCjDwZsvLSUG9G+Xq5GptXCL5/zb0NHj+XzWex1ZHttnPI73jfFeinNbwNd6AEFK7QVTB2IxhBiENgiarlgtG5Fvcfs3yBDOHmk0DohfwFHCEHLyzs7r9QEYuSJHkN2f67qM3UkEMR84hmCMyz2Dyu5sK+zkkyF5vxptGAhNfLXfZgdB5E6J2Trc10I6TzbWwaa9WOhhQkeB7HcPmAjlecdKxMXGGpkqTGfiUNNbIEeQcjHzBvfLSxNF8z6ght2VIXl2KX2lX3/mHMmnei6Z6VyM/rub1bMcbZWB6ITT/hK6eHZ/Sq3mbDZXUzsegz8rVIEu2XHL6DWIQ5AYoZkpj9ONJ0aLidzxxIGALJPE05sqrB/N7MchgXCCSOuIArE8qx9poROnsbTHwv9PdtER1yxYt6M8I3vEVOqv5RRwYQKcCnPxyaSKINC9Ur97KcSEGp6UhU1bQ75GOHTC+eRw6vzSyyYD80pqzBuSM7fSCwdjKpn3ZWufy/cWuOfLE4AWQgeMNqjy4BmgRah2hOnpBcbeS24RbRkV2ch/rNL3ayhDw/TBMUsUwBjD1gpKaA7v416DiR6ojwpTi6Sv6wcaSEoevhYfPwZPPdnCAtArtsz3rDeVG4vum8KM84Q1hBsQZRdipvdn00yw2/f7EiNgMZ8l5IKqR10XMrkY6fVnE2Ar5MmL2chBZy3wnNpJrxtH9TBIdGTi8jnqYQyJMpwzKUGi11Scsi2wzn0aPxB8OJ+v7l0Fky0nqEtwNCpQrNCckaIkQA6LirtPYSxSCIAjDVuLxBHkGPpAPpfrZzmOreNgf9+S2FhOsG9G+yoyj+2F1EVhb98aYE8xAPL0V8bIALxGRTH6r3A6EzoYfTmcbODTotBMTqpuqHSHHgpBUbylgw8uIkJ9HGTP6LUEraO996pz4ozQb+difKnpXYcNAj9uMtfXCfaSotRs4UiXQBY9E4Ih9cYjwTZgKgBKE+2oQ+kj0fczxqaH+k9dDNmay8CibrqeZLG8p7UMymUEgeiPG4B0NQRKdqhhX7wQHNlRXd/rTAkiGeu8BMOaAF3mBcDhuV3PunYZXPCXv49AqnNQv8upeM+gNxBq4J+zKt7lXmuCfDaRhgDl4bqj7gXqEV7fzvLoEPuXVm9p+FvCdxsfCfe9jNlwosLK3qkCkqv3I+wQpH2hWH5o3NuoI1aWUelwLQcDukf2oIpg5vn8Ig1iG0AJuY4E6WgEvQmybcNlqeOsOCtXm1Kwhlq9ZC9loUzE+ZlP0mU3lDduPzTvqTK8InclLicwLr5nWyg1eKdrysVW3n7cSJ04lwZYsNUCJcHis5gfV9hJiqDRDMBrdb0fBRHHzYi3wJS/cDsnfEwQC77UGmKSX1yLSgbGEb4LVIzSJz9OkclpZG0stW7qlAqkAEeanEqIiF1kU1yYdsgPcj3T5RqxCh4YUk59Vg+RoHRE7+X0hDpXx5ASn9PzuW7ZIKu62iNQVL2jInQ4QyENDRATyipgVGdYaoAyTi9k//+3/AggwweFQtQAA')))

for opponent, result in LOCAL_EVALUATION['cohorts'].items():
    games = [g for g in LOCAL_EVALUATION['games'] if g['opponent'] == opponent]
    assert len(games) == result['games'] and all(g['valid'] and g['margin'] == g['candidate_bank'] - g['opponent_bank'] for g in games)
    assert [sum(g['margin'] > 0 for g in games), sum(g['margin'] < 0 for g in games), sum(g['margin'] == 0 for g in games)] == [result['wins'], result['losses'], result['ties']]
    assert abs(sum(g['margin'] for g in games) / len(games) - result['mean_margin']) < 1e-8
    print(opponent, 'W/L/T', result['wins'], result['losses'], result['ties'], 'mean margin', round(result['mean_margin'], 1))


In [2]:
from pathlib import Path
import ast, base64, gzip, hashlib, io, tarfile, tempfile

WORK = Path.cwd()
ARCHIVE = WORK / "submission.tar.gz"
ARCHIVE_BYTES = base64.b64decode('...')
EXPECTED_ARCHIVE_SHA256 = "0ebc24aeca8ab1393ff70a5ed672a6023460c01e9fa51b29f389c9053167d8ba"
EXPECTED_MAIN_SHA256 = "4889137f1adf1f9266b2ad5a6ac6700c40e9be6dfe3654e39db375fb55b71341"
EXPECTED_NOTICE_SHA256 = "376a21d522d90aae03386b14d1dea815811ec06378c1cd7ac21450f9f02469a0"

def sha256(data):
    return hashlib.sha256(data).hexdigest()

assert sha256(ARCHIVE_BYTES) == EXPECTED_ARCHIVE_SHA256
ARCHIVE.write_bytes(ARCHIVE_BYTES)
RUNTIME_CLEANUP = tempfile.TemporaryDirectory(prefix="kg-agent-")
RUNTIME = Path(RUNTIME_CLEANUP.name)
with tarfile.open(ARCHIVE, "r:gz") as bundle:
    assert bundle.getnames() == ["LICENSE.txt", "NOTICE.txt", "main.py"]
    notice_bytes = bundle.extractfile("NOTICE.txt").read()
    main_bytes = bundle.extractfile("main.py").read()
assert sha256(main_bytes) == EXPECTED_MAIN_SHA256
assert sha256(notice_bytes) == EXPECTED_NOTICE_SHA256
compile(main_bytes, "main.py", "exec")
ast.parse(main_bytes)
MAIN = RUNTIME / "main.py"
NOTICE = RUNTIME / "NOTICE.txt"
MAIN.write_bytes(main_bytes)
NOTICE.write_bytes(notice_bytes)
print("PASS: byte-exact submission archive and preserved upstream files")
print("archive SHA-256:", EXPECTED_ARCHIVE_SHA256)
print("main.py SHA-256:", EXPECTED_MAIN_SHA256)
print("NOTICE.txt SHA-256:", EXPECTED_NOTICE_SHA256)

CONTROL_BYTES = main_bytes[:1032755]
assert sha256(CONTROL_BYTES) == EVALUATION_MANIFEST["opponent_source_sha256"]
CONTROL = RUNTIME / "original.py"
CONTROL.write_bytes(CONTROL_BYTES)

In [3]:
import base64, contextlib, hashlib, importlib.metadata, importlib.util, io, json, os, sys
from pathlib import Path

ENGINE_SOURCE = base64.b64decode("...")
ENGINE_SCHEMA = base64.b64decode("ewogICJuYW1lIjogImthZ2dyaWN1bHR1cmUiLAogICJ0aXRsZSI6ICJLYWdncmljdWx0dXJlIiwKICAiZGVzY3JpcHRpb24iOiAiQWR2YW5jZWQgZmFybWluZyBzaW11bGF0aW9uOiB0d28gcGxheWVycyBlYWNoIHRlbmQgYSAxMHgxMCBmYXJtIG9mIGZvdXIgNXg1IHF1YWRyYW50cywgZ3Jvd2luZyBjcm9wcywgcmFpc2luZyBhbmltYWxzLCBoaXJpbmcgaGVscCwgYW5kIHRyYWRpbmcgd2l0aCBhIGR5bmFtaWMgbWFya2V0IG92ZXIgb25lIHNlYXNvbi4iLAogICJ2ZXJzaW9uIjogIjAuMS4wIiwKICAiYWdlbnRzIjogWzJdLAogICJjb25maWd1cmF0aW9uIjogewogICAgImVwaXNvZGVTdGVwcyI6IDcyMCwKICAgICJhY3RUaW1lb3V0IjogMSwKICAgICJib2FyZFNpemUiOiB7CiAgICAgICJkZXNjcmlwdGlvbiI6ICJXaWR0aCBhbmQgaGVpZ2h0IChpbiB0aWxlcykgb2YgZWFjaCBwbGF5ZXIncyBzcXVhcmUgZmFybS4gQWR2YW5jZWQgZ2FtZSBleHBlY3RzIDEwIChmb3VyIDV4NSBxdWFkcmFudHMpLiIsCiAgICAgICJ0eXBlIjogImludGVnZXIiLAogICAgICAiZGVmYXVsdCI6IDEwLAogICAgICAibWluaW11bSI6IDQKICAgIH0sCiAgICAic3RhcnRpbmdNb25leSI6IHsKICAgICAgImRlc2NyaXB0aW9uIjogIk1vbmV5IGVhY2ggcGxheWVyIHN0YXJ0cyB3aXRoLiIsCiAgICAgICJ0eXBlIjogImludGVnZXIiLAogICAgICAiZGVmYXVsdCI6IDMwMDAsCiAgICAgICJtaW5pbXVtIjogMAogICAgfSwKICAgICJtYXhNYXJrZXRPcmRlcnNQZXJUdXJuIjogewogICAgICAiZGVzY3JpcHRpb24iOiAiTWF4aW11bSBudW1iZXIgb2YgbWFya2V0IG9yZGVycyBwcm9jZXNzZWQgcGVyIHBsYXllciBwZXIgdHVybi4gRXh0cmEgb3JkZXJzIGJleW9uZCB0aGlzIGxpbWl0IGFyZSBzaWxlbnRseSBkcm9wcGVkLiIsCiAgICAgICJ0eXBlIjogImludGVnZXIiLAogICAgICAiZGVmYXVsdCI6IDEwLAogICAgICAibWluaW11bSI6IDEKICAgIH0sCiAgICAidHVybnNQZXJEYXkiOiB7CiAgICAgICJkZXNjcmlwdGlvbiI6ICJOdW1iZXIgb2YgdHVybnMgdGhhdCBtYWtlIHVwIG9uZSBpbi1nYW1lIGRheS4iLAogICAgICAidHlwZSI6ICJpbnRlZ2VyIiwKICAgICAgImRlZmF1bHQiOiAyNCwKICAgICAgIm1pbmltdW0iOiAxCiAgICB9LAogICAgInNoZWRDYXBhY2l0eSI6IHsKICAgICAgImRlc2NyaXB0aW9uIjogIk1heGltdW0gbnVtYmVyIG9mIG5vbi1zZWVkIGl0ZW1zIHRoZSBzaGVkIGNhbiBob2xkIChvdmVyZmxvdyBhdCBlbmQtb2YtZGF5IGRyb3AgaXMgZGlzY2FyZGVkKS4iLAogICAgICAidHlwZSI6ICJpbnRlZ2VyIiwKICAgICAgImRlZmF1bHQiOiAxMDAsCiAgICAgICJtaW5pbXVtIjogMQogICAgfSwKICAgICJ3ZWVkU3Bhd25DaGFuY2UiOiB7CiAgICAgICJkZXNjcmlwdGlvbiI6ICJQZXItdGlsZSBwcm9iYWJpbGl0eSBvZiBhIHdlZWQgc3Bhd25pbmcgb24gYW4gZW1wdHkgdW5sb2NrZWQgdGlsZSBkdXJpbmcgdGhlIGVuZC1vZi1kYXkgcmVmcmVzaC4iLAogICAgICAidHlwZSI6ICJudW1iZXIiLAogICAgICAiZGVmYXVsdCI6IDAuMDA1LAogICAgICAibWluaW11bSI6IDAKICAgIH0sCiAgICAidG93blNob3BVbmxvY2tJbnRlcnZhbCI6IHsKICAgICAgImRlc2NyaXB0aW9uIjogIkRheXMgYmV0d2VlbiBzdWNjZXNzaXZlIHRvd24gc2hvcCB1bmxvY2tzLiBGaXJzdCBzaG9wIHVubG9ja3Mgb24gZGF5IGVxdWFsIHRvIHRoaXMgdmFsdWUuIFNob3BzIGFyZSBkcmF3biB3aXRoIHJlcGxhY2VtZW50LCBzbyB0aGUgc2FtZSBzaG9wIGNhbiB1bmxvY2sgbW9yZSB0aGFuIG9uY2U7IHVubG9ja2luZyBzdG9wcyBhZnRlciA4IGluc3RhbmNlcy4iLAogICAgICAidHlwZSI6ICJpbnRlZ2VyIiwKICAgICAgImRlZmF1bHQiOiAzLAogICAgICAibWluaW11bSI6IDEKICAgIH0sCiAgICAidG93blNob3BTZWxsSW50ZXJ2YWwiOiB7CiAgICAgICJkZXNjcmlwdGlvbiI6ICJOdW1iZXIgb2YgdHVybnMgYmV0d2VlbiBzdWNjZXNzaXZlIGNvbnN1bXB0aW9uIHRpY2tzIGJ5IGV2ZXJ5IHVubG9ja2VkIHRvd24gc2hvcCBpbnN0YW5jZS4gRWFjaCBpbnN0YW5jZSBwdWxscyBvbmUgb2YgZWFjaCBvZiBpdHMgcHJvZHVjdHMgcGVyIHRpY2sgKHNpbmdsZS1wcm9kdWN0IHNob3BzIHB1bGwgMngpLCBzbyBhIGR1cGxpY2F0ZWQgc2hvcCBjb25zdW1lcyBvbmNlIHBlciBjb3B5LiBUb3RhbCBkZW1hbmQgZ3Jvd3MgbW9ub3RvbmljYWxseSBhcyBtb3JlIHNob3BzIGFyZSB1bmxvY2tlZC4iLAogICAgICAidHlwZSI6ICJpbnRlZ2VyIiwKICAgICAgImRlZmF1bHQiOiA0LAogICAgICAibWluaW11bSI6IDEKICAgIH0sCiAgICAidG93bkNlbnRlclNlbGxJbnRlcnZhbCI6IHsKICAgICAgImRlc2NyaXB0aW9uIjogIk51bWJlciBvZiB0dXJucyBiZXR3ZWVuIHN1Y2Nlc3NpdmUgY29uc3VtcHRpb24gdGlja3MgYnkgdGhlIHRvd24gY2VudGVyIChvbmUgb2YgZXZlcnkgbm9uLWZlcnRpbGl6ZXIgcHJvZHVjdCBwZXIgdGljaykuIEF0IHRoZSBkZWZhdWx0IG9mIDI0IHdpdGggdHVybnNQZXJEYXkgMjQsIHRoZSB0b3duIGNlbnRlciBidXlzIG9uY2UgcGVyIGRheSBhdCBhIGZsYXQgcmF0ZSBmb3IgdGhlIHdob2xlIHNlYXNvbi4iLAogICAgICAidHlwZSI6ICJpbnRlZ2VyIiwKICAgICAgImRlZmF1bHQiOiAyNCwKICAgICAgIm1pbmltdW0iOiAxCiAgICB9LAogICAgInNlZWQiOiB7CiAgICAgICJkZXNjcmlwdGlvbiI6ICJPcHRpb25hbCBpbnB1dCBzZWVkIGZvciBkZXRlcm1pbmlzdGljIGVwaXNvZGUgZ2VuZXJhdGlvbi4gVGhlIGludGVycHJldGVyIGNsZWFycyB0aGlzIGZyb20gY29uZmlndXJhdGlvbiBhZnRlciByZWFkaW5nIGFuZCBzdG9yZXMgdGhlIHJlc29sdmVkIHNlZWQgb24gZW52LmluZm9bJ3NlZWQnXSBzbyBpdCBzdGF5cyBvdXQgb2YgYWdlbnQgb2JzZXJ2YXRpb25zIGJ1dCBpcyBzdGlsbCByZWNvcmRlZCBpbiB0aGUgcmVwbGF5LiIsCiAgICAgICJ0eXBlIjogWyJpbnRlZ2VyIiwgIm51bGwiXSwKICAgICAgImRlZmF1bHQiOiBudWxsCiAgICB9LAogICAgImZhcm1IYW5kQ29zdE11bHQiOiB7CiAgICAgICJkZXNjcmlwdGlvbiI6ICJNdWx0aXBsaWVyIGFwcGxpZWQgdG8gdGhlIEZpYm9uYWNjaSBoaXJlLWNvc3Qgc2VxdWVuY2UgKDEsIDEsIDIsIDMsIDUsIDgsIDEzLCAuLi4pLiBUaGUgbi10aCBoaXJlIG9mIHRoZSBkYXkgY29zdHMgdGhpcyB2YWx1ZSB0aW1lcyBmaWIobikuIiwKICAgICAgInR5cGUiOiAiaW50ZWdlciIsCiAgICAgICJkZWZhdWx0IjogMSwKICAgICAgIm1pbmltdW0iOiAwCiAgICB9LAogICAgIm1hcmtldFBhcmFtcyI6IHsKICAgICAgImRlc2NyaXB0aW9uIjogIlBlci1yZXNvdXJjZSBtYXJrZXQgcHJpY2UtY3VydmUgb3ZlcnJpZGVzLiBTcGFyc2U6IGtleWVkIGJ5IHByb2R1Y3QgbmFtZTsgZWFjaCBlbnRyeSBtYXkgc2V0IGFueSBzdWJzZXQgb2Yge2Jhc2UsIEkwLCBULCBiZWxvd19mdW5jLCBiZWxvd190YXJnZXQsIGFib3ZlX2Z1bmMsIGFib3ZlX3RhcmdldH0uIE1pc3Npbmcga2V5cyAoYW5kIG1pc3NpbmcgcHJvZHVjdHMpIGluaGVyaXQgZnJvbSBNQVJLRVRfUEFSQU1TIGRlZmF1bHRzIGluIGthZ2dyaWN1bHR1cmUucHkuIEZ1bmN0aW9uczogbGluZWFyLCBzcSwgc3FydCwgbG9nLCBsb2cxMCwgaGluZ2UuIGFtcCBpcyBkZXJpdmVkIGFzIHRhcmdldCAqIGJhc2UgLyBmKFQpLiBUaGUgaGluZ2UgZnVuY3Rpb24gaXMgZmxhdC1pc2ggYmVsb3cgVCBhbmQgc3Bpa2VzIGFib3ZlIGl0LCBhbmQgdW5saWtlIHRoZSBvdGhlcnMgaXQgaXMgc2NhbGVkIGJ5IFQgc28gdGhhdCBmKFQpID0gMS4iLAogICAgICAidHlwZSI6ICJvYmplY3QiLAogICAgICAiZGVmYXVsdCI6IHt9CiAgICB9CiAgfSwKICAicmV3YXJkIjogewogICAgImRlc2NyaXB0aW9uIjogIlBsYXllciBtb25leSBhdCBlbmQgb2YgZ2FtZSAoZmluYWwgc2NvcmUpLiIsCiAgICAidHlwZSI6ICJudW1iZXIiLAogICAgImRlZmF1bHQiOiAwCiAgfSwKICAib2JzZXJ2YXRpb24iOiB7CiAgICAicGxheWVyIjogewogICAgICAiZGVzY3JpcHRpb24iOiAiUGxheWVyIElEICgwIG9yIDEpLiIsCiAgICAgICJ0eXBlIjogImludGVnZXIiLAogICAgICAiZGVmYXVsdCI6IDAKICAgIH0sCiAgICAiZmFybXMiOiB7CiAgICAgICJkZXNjcmlwdGlvbiI6ICJQZXItcGxheWVyIHB1YmxpYyBmYXJtIHN0YXRlIHZpc2libGUgdG8gYWxsIGFnZW50cyAodGlsZXMsIG1vbmV5LCBmYXJtZXIvaGFuZCBwb3NpdGlvbnMsIHVubG9ja2VkIHF1YWRyYW50cywgaGlyZSBjb3VudCkuIEluZGV4ZWQgYnkgcGxheWVyIGlkLiBPcHBvbmVudCBzaGVkIGFuZCBwZXItZmFybWVyIGludmVudG9yaWVzIGFyZSBOT1QgaW5jbHVkZWQgaGVyZS4iLAogICAgICAidHlwZSI6ICJhcnJheSIsCiAgICAgICJzaGFyZWQiOiB0cnVlLAogICAgICAiZGVmYXVsdCI6IFtdCiAgICB9LAogICAgInByaXZhdGUiOiB7CiAgICAgICJkZXNjcmlwdGlvbiI6ICJQZXItYWdlbnQgcHJpdmF0ZSBzdGF0ZSBmb3IgVEhJUyBwbGF5ZXIgb25seTogc2hlZCBpbnZlbnRvcnksIHBlci1mYXJtZXIgaW52ZW50b3JpZXMsIHNlZWQgY291bnRzLiBOb3Qgc2hhcmVkLiIsCiAgICAgICJ0eXBlIjogIm9iamVjdCIsCiAgICAgICJzaGFyZWQiOiBmYWxzZSwKICAgICAgImRlZmF1bHQiOiB7fQogICAgfSwKICAgICJtYXJrZXQiOiB7CiAgICAgICJkZXNjcmlwdGlvbiI6ICJTaGFyZWQgbWFya2V0IHN0YXRlOiBjdXJyZW50IHBlci1wcm9kdWN0IGludmVudG9yeSBhbmQgcHJpY2UuIiwKICAgICAgInR5cGUiOiAib2JqZWN0IiwKICAgICAgInNoYXJlZCI6IHRydWUsCiAgICAgICJkZWZhdWx0Ijoge30KICAgIH0sCiAgICAidG93biI6IHsKICAgICAgImRlc2NyaXB0aW9uIjogIlNoYXJlZCB0b3duIHN0YXRlOiBsaXN0IG9mIHVubG9ja2VkIHNob3AgbmFtZXMuIEVhY2ggdW5sb2NrZWQgc2hvcCBnZW5lcmF0ZXMgcGVyLXRpY2sgZGVtYW5kIGZvciBpdHMgcHJvZHVjdHMuIiwKICAgICAgInR5cGUiOiAib2JqZWN0IiwKICAgICAgInNoYXJlZCI6IHRydWUsCiAgICAgICJkZWZhdWx0Ijoge30KICAgIH0sCiAgICAiZGF5IjogewogICAgICAiZGVzY3JpcHRpb24iOiAiQ3VycmVudCBpbi1nYW1lIGRheSAoMC1pbmRleGVkKS4iLAogICAgICAidHlwZSI6ICJpbnRlZ2VyIiwKICAgICAgInNoYXJlZCI6IHRydWUsCiAgICAgICJkZWZhdWx0IjogMAogICAgfSwKICAgICJob3VyIjogewogICAgICAiZGVzY3JpcHRpb24iOiAiQ3VycmVudCB0dXJuIHdpdGhpbiB0aGUgZGF5ICgwLWluZGV4ZWQpLiIsCiAgICAgICJ0eXBlIjogImludGVnZXIiLAogICAgICAic2hhcmVkIjogdHJ1ZSwKICAgICAgImRlZmF1bHQiOiAwCiAgICB9LAogICAgInJlbWFpbmluZ092ZXJhZ2VUaW1lIjogNjAKICB9LAogICJhY3Rpb24iOiB7CiAgICAiZGVzY3JpcHRpb24iOiAiUGVyLXR1cm4gYWN0aW9uIG9mIHRoZSBmb3JtIHtcImZhcm1lclwiOiBbb3AsIC4uLmFyZ3NdLCBcImhhbmRzXCI6IFtbb3AsIC4uLmFyZ3NdLCAuLi5dLCBcIm1hcmtldFwiOiBbW29wLCAuLi5hcmdzXSwgLi4uXX0uIEZhcm1lci9oYW5kIG9wczogTk9SVEgsIFNPVVRILCBFQVNULCBXRVNULCBQQVNTLCBQSUNLVVAgPGl0ZW0+IFtuXSwgUExBTlQgPGNyb3A+LCBXQVRFUiwgSEFSVkVTVCwgRkVSVElMSVpFLCBCVUlMRF9DT09QLCBCVUlMRF9QQVNUVVJFLCBESUcsIFBMQUNFIDxpdGVtPiBbbl0sIEZFRUQsIENPTExFQ1RfRkVSVElMSVpFUiwgQ0FSRS4gTWFya2V0IG9wczogQlVZX1NFRUQgPGNyb3A+IDxuPiwgQlVZX1BST0RVQ1QgPGl0ZW0+IDxuPiwgQlVZX0FOSU1BTCA8YW5pbWFsPiA8bj4sIFNFTEwgPGl0ZW0+IDxuPiwgSElSRSwgQlVZX0xBTkQuIiwKICAgICJ0eXBlIjogIm9iamVjdCIsCiAgICAiZGVmYXVsdCI6IHsgImZhcm1lciI6IFsiUEFTUyJdLCAiaGFuZHMiOiBbXSwgIm1hcmtldCI6IFtdIH0KICB9LAogICJzdGF0dXMiOiB7CiAgICAiZGVmYXVsdHMiOiBbIkFDVElWRSIsICJBQ1RJVkUiXQogIH0KfQo=")
ENGINE_LICENSE = base64.b64decode("ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgQXBhY2hlIExpY2Vuc2UKICAgICAgICAgICAgICAgICAgICAgICAgICAgVmVyc2lvbiAyLjAsIEphbnVhcnkgMjAwNAogICAgICAgICAgICAgICAgICAgICAgICBodHRwOi8vd3d3LmFwYWNoZS5vcmcvbGljZW5zZXMvCgogICBURVJNUyBBTkQgQ09ORElUSU9OUyBGT1IgVVNFLCBSRVBST0RVQ1RJT04sIEFORCBESVNUUklCVVRJT04KCiAgIDEuIERlZmluaXRpb25zLgoKICAgICAgIkxpY2Vuc2UiIHNoYWxsIG1lYW4gdGhlIHRlcm1zIGFuZCBjb25kaXRpb25zIGZvciB1c2UsIHJlcHJvZHVjdGlvbiwKICAgICAgYW5kIGRpc3RyaWJ1dGlvbiBhcyBkZWZpbmVkIGJ5IFNlY3Rpb25zIDEgdGhyb3VnaCA5IG9mIHRoaXMgZG9jdW1lbnQuCgogICAgICAiTGljZW5zb3IiIHNoYWxsIG1lYW4gdGhlIGNvcHlyaWdodCBvd25lciBvciBlbnRpdHkgYXV0aG9yaXplZCBieQogICAgICB0aGUgY29weXJpZ2h0IG93bmVyIHRoYXQgaXMgZ3JhbnRpbmcgdGhlIExpY2Vuc2UuCgogICAgICAiTGVnYWwgRW50aXR5IiBzaGFsbCBtZWFuIHRoZSB1bmlvbiBvZiB0aGUgYWN0aW5nIGVudGl0eSBhbmQgYWxsCiAgICAgIG90aGVyIGVudGl0aWVzIHRoYXQgY29udHJvbCwgYXJlIGNvbnRyb2xsZWQgYnksIG9yIGFyZSB1bmRlciBjb21tb24KICAgICAgY29udHJvbCB3aXRoIHRoYXQgZW50aXR5LiBGb3IgdGhlIHB1cnBvc2VzIG9mIHRoaXMgZGVmaW5pdGlvbiwKICAgICAgImNvbnRyb2wiIG1lYW5zIChpKSB0aGUgcG93ZXIsIGRpcmVjdCBvciBpbmRpcmVjdCwgdG8gY2F1c2UgdGhlCiAgICAgIGRpcmVjdGlvbiBvciBtYW5hZ2VtZW50IG9mIHN1Y2ggZW50aXR5LCB3aGV0aGVyIGJ5IGNvbnRyYWN0IG9yCiAgICAgIG90aGVyd2lzZSwgb3IgKGlpKSBvd25lcnNoaXAgb2YgZmlmdHkgcGVyY2VudCAoNTAlKSBvciBtb3JlIG9mIHRoZQogICAgICBvdXRzdGFuZGluZyBzaGFyZXMsIG9yIChpaWkpIGJlbmVmaWNpYWwgb3duZXJzaGlwIG9mIHN1Y2ggZW50aXR5LgoKICAgICAgIllvdSIgKG9yICJZb3VyIikgc2hhbGwgbWVhbiBhbiBpbmRpdmlkdWFsIG9yIExlZ2FsIEVudGl0eQogICAgICBleGVyY2lzaW5nIHBlcm1pc3Npb25zIGdyYW50ZWQgYnkgdGhpcyBMaWNlbnNlLgoKICAgICAgIlNvdXJjZSIgZm9ybSBzaGFsbCBtZWFuIHRoZSBwcmVmZXJyZWQgZm9ybSBmb3IgbWFraW5nIG1vZGlmaWNhdGlvbnMsCiAgICAgIGluY2x1ZGluZyBidXQgbm90IGxpbWl0ZWQgdG8gc29mdHdhcmUgc291cmNlIGNvZGUsIGRvY3VtZW50YXRpb24KICAgICAgc291cmNlLCBhbmQgY29uZmlndXJhdGlvbiBmaWxlcy4KCiAgICAgICJPYmplY3QiIGZvcm0gc2hhbGwgbWVhbiBhbnkgZm9ybSByZXN1bHRpbmcgZnJvbSBtZWNoYW5pY2FsCiAgICAgIHRyYW5zZm9ybWF0aW9uIG9yIHRyYW5zbGF0aW9uIG9mIGEgU291cmNlIGZvcm0sIGluY2x1ZGluZyBidXQKICAgICAgbm90IGxpbWl0ZWQgdG8gY29tcGlsZWQgb2JqZWN0IGNvZGUsIGdlbmVyYXRlZCBkb2N1bWVudGF0aW9uLAogICAgICBhbmQgY29udmVyc2lvbnMgdG8gb3RoZXIgbWVkaWEgdHlwZXMuCgogICAgICAiV29yayIgc2hhbGwgbWVhbiB0aGUgd29yayBvZiBhdXRob3JzaGlwLCB3aGV0aGVyIGluIFNvdXJjZSBvcgogICAgICBPYmplY3QgZm9ybSwgbWFkZSBhdmFpbGFibGUgdW5kZXIgdGhlIExpY2Vuc2UsIGFzIGluZGljYXRlZCBieSBhCiAgICAgIGNvcHlyaWdodCBub3RpY2UgdGhhdCBpcyBpbmNsdWRlZCBpbiBvciBhdHRhY2hlZCB0byB0aGUgd29yawogICAgICAoYW4gZXhhbXBsZSBpcyBwcm92aWRlZCBpbiB0aGUgQXBwZW5kaXggYmVsb3cpLgoKICAgICAgIkRlcml2YXRpdmUgV29ya3MiIHNoYWxsIG1lYW4gYW55IHdvcmssIHdoZXRoZXIgaW4gU291cmNlIG9yIE9iamVjdAogICAgICBmb3JtLCB0aGF0IGlzIGJhc2VkIG9uIChvciBkZXJpdmVkIGZyb20pIHRoZSBXb3JrIGFuZCBmb3Igd2hpY2ggdGhlCiAgICAgIGVkaXRvcmlhbCByZXZpc2lvbnMsIGFubm90YXRpb25zLCBlbGFib3JhdGlvbnMsIG9yIG90aGVyIG1vZGlmaWNhdGlvbnMKICAgICAgcmVwcmVzZW50LCBhcyBhIHdob2xlLCBhbiBvcmlnaW5hbCB3b3JrIG9mIGF1dGhvcnNoaXAuIEZvciB0aGUgcHVycG9zZXMKICAgICAgb2YgdGhpcyBMaWNlbnNlLCBEZXJpdmF0aXZlIFdvcmtzIHNoYWxsIG5vdCBpbmNsdWRlIHdvcmtzIHRoYXQgcmVtYWluCiAgICAgIHNlcGFyYWJsZSBmcm9tLCBvciBtZXJlbHkgbGluayAob3IgYmluZCBieSBuYW1lKSB0byB0aGUgaW50ZXJmYWNlcyBvZiwKICAgICAgdGhlIFdvcmsgYW5kIERlcml2YXRpdmUgV29ya3MgdGhlcmVvZi4KCiAgICAgICJDb250cmlidXRpb24iIHNoYWxsIG1lYW4gYW55IHdvcmsgb2YgYXV0aG9yc2hpcCwgaW5jbHVkaW5nCiAgICAgIHRoZSBvcmlnaW5hbCB2ZXJzaW9uIG9mIHRoZSBXb3JrIGFuZCBhbnkgbW9kaWZpY2F0aW9ucyBvciBhZGRpdGlvbnMKICAgICAgdG8gdGhhdCBXb3JrIG9yIERlcml2YXRpdmUgV29ya3MgdGhlcmVvZiwgdGhhdCBpcyBpbnRlbnRpb25hbGx5CiAgICAgIHN1Ym1pdHRlZCB0byBMaWNlbnNvciBmb3IgaW5jbHVzaW9uIGluIHRoZSBXb3JrIGJ5IHRoZSBjb3B5cmlnaHQgb3duZXIKICAgICAgb3IgYnkgYW4gaW5kaXZpZHVhbCBvciBMZWdhbCBFbnRpdHkgYXV0aG9yaXplZCB0byBzdWJtaXQgb24gYmVoYWxmIG9mCiAgICAgIHRoZSBjb3B5cmlnaHQgb3duZXIuIEZvciB0aGUgcHVycG9zZXMgb2YgdGhpcyBkZWZpbml0aW9uLCAic3VibWl0dGVkIgogICAgICBtZWFucyBhbnkgZm9ybSBvZiBlbGVjdHJvbmljLCB2ZXJiYWwsIG9yIHdyaXR0ZW4gY29tbXVuaWNhdGlvbiBzZW50CiAgICAgIHRvIHRoZSBMaWNlbnNvciBvciBpdHMgcmVwcmVzZW50YXRpdmVzLCBpbmNsdWRpbmcgYnV0IG5vdCBsaW1pdGVkIHRvCiAgICAgIGNvbW11bmljYXRpb24gb24gZWxlY3Ryb25pYyBtYWlsaW5nIGxpc3RzLCBzb3VyY2UgY29kZSBjb250cm9sIHN5c3RlbXMsCiAgICAgIGFuZCBpc3N1ZSB0cmFja2luZyBzeXN0ZW1zIHRoYXQgYXJlIG1hbmFnZWQgYnksIG9yIG9uIGJlaGFsZiBvZiwgdGhlCiAgICAgIExpY2Vuc29yIGZvciB0aGUgcHVycG9zZSBvZiBkaXNjdXNzaW5nIGFuZCBpbXByb3ZpbmcgdGhlIFdvcmssIGJ1dAogICAgICBleGNsdWRpbmcgY29tbXVuaWNhdGlvbiB0aGF0IGlzIGNvbnNwaWN1b3VzbHkgbWFya2VkIG9yIG90aGVyd2lzZQogICAgICBkZXNpZ25hdGVkIGluIHdyaXRpbmcgYnkgdGhlIGNvcHlyaWdodCBvd25lciBhcyAiTm90IGEgQ29udHJpYnV0aW9uLiIKCiAgICAgICJDb250cmlidXRvciIgc2hhbGwgbWVhbiBMaWNlbnNvciBhbmQgYW55IGluZGl2aWR1YWwgb3IgTGVnYWwgRW50aXR5CiAgICAgIG9uIGJlaGFsZiBvZiB3aG9tIGEgQ29udHJpYnV0aW9uIGhhcyBiZWVuIHJlY2VpdmVkIGJ5IExpY2Vuc29yIGFuZAogICAgICBzdWJzZXF1ZW50bHkgaW5jb3Jwb3JhdGVkIHdpdGhpbiB0aGUgV29yay4KCiAgIDIuIEdyYW50IG9mIENvcHlyaWdodCBMaWNlbnNlLiBTdWJqZWN0IHRvIHRoZSB0ZXJtcyBhbmQgY29uZGl0aW9ucyBvZgogICAgICB0aGlzIExpY2Vuc2UsIGVhY2ggQ29udHJpYnV0b3IgaGVyZWJ5IGdyYW50cyB0byBZb3UgYSBwZXJwZXR1YWwsCiAgICAgIHdvcmxkd2lkZSwgbm9uLWV4Y2x1c2l2ZSwgbm8tY2hhcmdlLCByb3lhbHR5LWZyZWUsIGlycmV2b2NhYmxlCiAgICAgIGNvcHlyaWdodCBsaWNlbnNlIHRvIHJlcHJvZHVjZSwgcHJlcGFyZSBEZXJpdmF0aXZlIFdvcmtzIG9mLAogICAgICBwdWJsaWNseSBkaXNwbGF5LCBwdWJsaWNseSBwZXJmb3JtLCBzdWJsaWNlbnNlLCBhbmQgZGlzdHJpYnV0ZSB0aGUKICAgICAgV29yayBhbmQgc3VjaCBEZXJpdmF0aXZlIFdvcmtzIGluIFNvdXJjZSBvciBPYmplY3QgZm9ybS4KCiAgIDMuIEdyYW50IG9mIFBhdGVudCBMaWNlbnNlLiBTdWJqZWN0IHRvIHRoZSB0ZXJtcyBhbmQgY29uZGl0aW9ucyBvZgogICAgICB0aGlzIExpY2Vuc2UsIGVhY2ggQ29udHJpYnV0b3IgaGVyZWJ5IGdyYW50cyB0byBZb3UgYSBwZXJwZXR1YWwsCiAgICAgIHdvcmxkd2lkZSwgbm9uLWV4Y2x1c2l2ZSwgbm8tY2hhcmdlLCByb3lhbHR5LWZyZWUsIGlycmV2b2NhYmxlCiAgICAgIChleGNlcHQgYXMgc3RhdGVkIGluIHRoaXMgc2VjdGlvbikgcGF0ZW50IGxpY2Vuc2UgdG8gbWFrZSwgaGF2ZSBtYWRlLAogICAgICB1c2UsIG9mZmVyIHRvIHNlbGwsIHNlbGwsIGltcG9ydCwgYW5kIG90aGVyd2lzZSB0cmFuc2ZlciB0aGUgV29yaywKICAgICAgd2hlcmUgc3VjaCBsaWNlbnNlIGFwcGxpZXMgb25seSB0byB0aG9zZSBwYXRlbnQgY2xhaW1zIGxpY2Vuc2FibGUKICAgICAgYnkgc3VjaCBDb250cmlidXRvciB0aGF0IGFyZSBuZWNlc3NhcmlseSBpbmZyaW5nZWQgYnkgdGhlaXIKICAgICAgQ29udHJpYnV0aW9uKHMpIGFsb25lIG9yIGJ5IGNvbWJpbmF0aW9uIG9mIHRoZWlyIENvbnRyaWJ1dGlvbihzKQogICAgICB3aXRoIHRoZSBXb3JrIHRvIHdoaWNoIHN1Y2ggQ29udHJpYnV0aW9uKHMpIHdhcyBzdWJtaXR0ZWQuIElmIFlvdQogICAgICBpbnN0aXR1dGUgcGF0ZW50IGxpdGlnYXRpb24gYWdhaW5zdCBhbnkgZW50aXR5IChpbmNsdWRpbmcgYQogICAgICBjcm9zcy1jbGFpbSBvciBjb3VudGVyY2xhaW0gaW4gYSBsYXdzdWl0KSBhbGxlZ2luZyB0aGF0IHRoZSBXb3JrCiAgICAgIG9yIGEgQ29udHJpYnV0aW9uIGluY29ycG9yYXRlZCB3aXRoaW4gdGhlIFdvcmsgY29uc3RpdHV0ZXMgZGlyZWN0CiAgICAgIG9yIGNvbnRyaWJ1dG9yeSBwYXRlbnQgaW5mcmluZ2VtZW50LCB0aGVuIGFueSBwYXRlbnQgbGljZW5zZXMKICAgICAgZ3JhbnRlZCB0byBZb3UgdW5kZXIgdGhpcyBMaWNlbnNlIGZvciB0aGF0IFdvcmsgc2hhbGwgdGVybWluYXRlCiAgICAgIGFzIG9mIHRoZSBkYXRlIHN1Y2ggbGl0aWdhdGlvbiBpcyBmaWxlZC4KCiAgIDQuIFJlZGlzdHJpYnV0aW9uLiBZb3UgbWF5IHJlcHJvZHVjZSBhbmQgZGlzdHJpYnV0ZSBjb3BpZXMgb2YgdGhlCiAgICAgIFdvcmsgb3IgRGVyaXZhdGl2ZSBXb3JrcyB0aGVyZW9mIGluIGFueSBtZWRpdW0sIHdpdGggb3Igd2l0aG91dAogICAgICBtb2RpZmljYXRpb25zLCBhbmQgaW4gU291cmNlIG9yIE9iamVjdCBmb3JtLCBwcm92aWRlZCB0aGF0IFlvdQogICAgICBtZWV0IHRoZSBmb2xsb3dpbmcgY29uZGl0aW9uczoKCiAgICAgIChhKSBZb3UgbXVzdCBnaXZlIGFueSBvdGhlciByZWNpcGllbnRzIG9mIHRoZSBXb3JrIG9yCiAgICAgICAgICBEZXJpdmF0aXZlIFdvcmtzIGEgY29weSBvZiB0aGlzIExpY2Vuc2U7IGFuZAoKICAgICAgKGIpIFlvdSBtdXN0IGNhdXNlIGFueSBtb2RpZmllZCBmaWxlcyB0byBjYXJyeSBwcm9taW5lbnQgbm90aWNlcwogICAgICAgICAgc3RhdGluZyB0aGF0IFlvdSBjaGFuZ2VkIHRoZSBmaWxlczsgYW5kCgogICAgICAoYykgWW91IG11c3QgcmV0YWluLCBpbiB0aGUgU291cmNlIGZvcm0gb2YgYW55IERlcml2YXRpdmUgV29ya3MKICAgICAgICAgIHRoYXQgWW91IGRpc3RyaWJ1dGUsIGFsbCBjb3B5cmlnaHQsIHBhdGVudCwgdHJhZGVtYXJrLCBhbmQKICAgICAgICAgIGF0dHJpYnV0aW9uIG5vdGljZXMgZnJvbSB0aGUgU291cmNlIGZvcm0gb2YgdGhlIFdvcmssCiAgICAgICAgICBleGNsdWRpbmcgdGhvc2Ugbm90aWNlcyB0aGF0IGRvIG5vdCBwZXJ0YWluIHRvIGFueSBwYXJ0IG9mCiAgICAgICAgICB0aGUgRGVyaXZhdGl2ZSBXb3JrczsgYW5kCgogICAgICAoZCkgSWYgdGhlIFdvcmsgaW5jbHVkZXMgYSAiTk9USUNFIiB0ZXh0IGZpbGUgYXMgcGFydCBvZiBpdHMKICAgICAgICAgIGRpc3RyaWJ1dGlvbiwgdGhlbiBhbnkgRGVyaXZhdGl2ZSBXb3JrcyB0aGF0IFlvdSBkaXN0cmlidXRlIG11c3QKICAgICAgICAgIGluY2x1ZGUgYSByZWFkYWJsZSBjb3B5IG9mIHRoZSBhdHRyaWJ1dGlvbiBub3RpY2VzIGNvbnRhaW5lZAogICAgICAgICAgd2l0aGluIHN1Y2ggTk9USUNFIGZpbGUsIGV4Y2x1ZGluZyB0aG9zZSBub3RpY2VzIHRoYXQgZG8gbm90CiAgICAgICAgICBwZXJ0YWluIHRvIGFueSBwYXJ0IG9mIHRoZSBEZXJpdmF0aXZlIFdvcmtzLCBpbiBhdCBsZWFzdCBvbmUKICAgICAgICAgIG9mIHRoZSBmb2xsb3dpbmcgcGxhY2VzOiB3aXRoaW4gYSBOT1RJQ0UgdGV4dCBmaWxlIGRpc3RyaWJ1dGVkCiAgICAgICAgICBhcyBwYXJ0IG9mIHRoZSBEZXJpdmF0aXZlIFdvcmtzOyB3aXRoaW4gdGhlIFNvdXJjZSBmb3JtIG9yCiAgICAgICAgICBkb2N1bWVudGF0aW9uLCBpZiBwcm92aWRlZCBhbG9uZyB3aXRoIHRoZSBEZXJpdmF0aXZlIFdvcmtzOyBvciwKICAgICAgICAgIHdpdGhpbiBhIGRpc3BsYXkgZ2VuZXJhdGVkIGJ5IHRoZSBEZXJpdmF0aXZlIFdvcmtzLCBpZiBhbmQKICAgICAgICAgIHdoZXJldmVyIHN1Y2ggdGhpcmQtcGFydHkgbm90aWNlcyBub3JtYWxseSBhcHBlYXIuIFRoZSBjb250ZW50cwogICAgICAgICAgb2YgdGhlIE5PVElDRSBmaWxlIGFyZSBmb3IgaW5mb3JtYXRpb25hbCBwdXJwb3NlcyBvbmx5IGFuZAogICAgICAgICAgZG8gbm90IG1vZGlmeSB0aGUgTGljZW5zZS4gWW91IG1heSBhZGQgWW91ciBvd24gYXR0cmlidXRpb24KICAgICAgICAgIG5vdGljZXMgd2l0aGluIERlcml2YXRpdmUgV29ya3MgdGhhdCBZb3UgZGlzdHJpYnV0ZSwgYWxvbmdzaWRlCiAgICAgICAgICBvciBhcyBhbiBhZGRlbmR1bSB0byB0aGUgTk9USUNFIHRleHQgZnJvbSB0aGUgV29yaywgcHJvdmlkZWQKICAgICAgICAgIHRoYXQgc3VjaCBhZGRpdGlvbmFsIGF0dHJpYnV0aW9uIG5vdGljZXMgY2Fubm90IGJlIGNvbnN0cnVlZAogICAgICAgICAgYXMgbW9kaWZ5aW5nIHRoZSBMaWNlbnNlLgoKICAgICAgWW91IG1heSBhZGQgWW91ciBvd24gY29weXJpZ2h0IHN0YXRlbWVudCB0byBZb3VyIG1vZGlmaWNhdGlvbnMgYW5kCiAgICAgIG1heSBwcm92aWRlIGFkZGl0aW9uYWwgb3IgZGlmZmVyZW50IGxpY2Vuc2UgdGVybXMgYW5kIGNvbmRpdGlvbnMKICAgICAgZm9yIHVzZSwgcmVwcm9kdWN0aW9uLCBvciBkaXN0cmlidXRpb24gb2YgWW91ciBtb2RpZmljYXRpb25zLCBvcgogICAgICBmb3IgYW55IHN1Y2ggRGVyaXZhdGl2ZSBXb3JrcyBhcyBhIHdob2xlLCBwcm92aWRlZCBZb3VyIHVzZSwKICAgICAgcmVwcm9kdWN0aW9uLCBhbmQgZGlzdHJpYnV0aW9uIG9mIHRoZSBXb3JrIG90aGVyd2lzZSBjb21wbGllcyB3aXRoCiAgICAgIHRoZSBjb25kaXRpb25zIHN0YXRlZCBpbiB0aGlzIExpY2Vuc2UuCgogICA1LiBTdWJtaXNzaW9uIG9mIENvbnRyaWJ1dGlvbnMuIFVubGVzcyBZb3UgZXhwbGljaXRseSBzdGF0ZSBvdGhlcndpc2UsCiAgICAgIGFueSBDb250cmlidXRpb24gaW50ZW50aW9uYWxseSBzdWJtaXR0ZWQgZm9yIGluY2x1c2lvbiBpbiB0aGUgV29yawogICAgICBieSBZb3UgdG8gdGhlIExpY2Vuc29yIHNoYWxsIGJlIHVuZGVyIHRoZSB0ZXJtcyBhbmQgY29uZGl0aW9ucyBvZgogICAgICB0aGlzIExpY2Vuc2UsIHdpdGhvdXQgYW55IGFkZGl0aW9uYWwgdGVybXMgb3IgY29uZGl0aW9ucy4KICAgICAgTm90d2l0aHN0YW5kaW5nIHRoZSBhYm92ZSwgbm90aGluZyBoZXJlaW4gc2hhbGwgc3VwZXJzZWRlIG9yIG1vZGlmeQogICAgICB0aGUgdGVybXMgb2YgYW55IHNlcGFyYXRlIGxpY2Vuc2UgYWdyZWVtZW50IHlvdSBtYXkgaGF2ZSBleGVjdXRlZAogICAgICB3aXRoIExpY2Vuc29yIHJlZ2FyZGluZyBzdWNoIENvbnRyaWJ1dGlvbnMuCgogICA2LiBUcmFkZW1hcmtzLiBUaGlzIExpY2Vuc2UgZG9lcyBub3QgZ3JhbnQgcGVybWlzc2lvbiB0byB1c2UgdGhlIHRyYWRlCiAgICAgIG5hbWVzLCB0cmFkZW1hcmtzLCBzZXJ2aWNlIG1hcmtzLCBvciBwcm9kdWN0IG5hbWVzIG9mIHRoZSBMaWNlbnNvciwKICAgICAgZXhjZXB0IGFzIHJlcXVpcmVkIGZvciByZWFzb25hYmxlIGFuZCBjdXN0b21hcnkgdXNlIGluIGRlc2NyaWJpbmcgdGhlCiAgICAgIG9yaWdpbiBvZiB0aGUgV29yayBhbmQgcmVwcm9kdWNpbmcgdGhlIGNvbnRlbnQgb2YgdGhlIE5PVElDRSBmaWxlLgoKICAgNy4gRGlzY2xhaW1lciBvZiBXYXJyYW50eS4gVW5sZXNzIHJlcXVpcmVkIGJ5IGFwcGxpY2FibGUgbGF3IG9yCiAgICAgIGFncmVlZCB0byBpbiB3cml0aW5nLCBMaWNlbnNvciBwcm92aWRlcyB0aGUgV29yayAoYW5kIGVhY2gKICAgICAgQ29udHJpYnV0b3IgcHJvdmlkZXMgaXRzIENvbnRyaWJ1dGlvbnMpIG9uIGFuICJBUyBJUyIgQkFTSVMsCiAgICAgIFdJVEhPVVQgV0FSUkFOVElFUyBPUiBDT05ESVRJT05TIE9GIEFOWSBLSU5ELCBlaXRoZXIgZXhwcmVzcyBvcgogICAgICBpbXBsaWVkLCBpbmNsdWRpbmcsIHdpdGhvdXQgbGltaXRhdGlvbiwgYW55IHdhcnJhbnRpZXMgb3IgY29uZGl0aW9ucwogICAgICBvZiBUSVRMRSwgTk9OLUlORlJJTkdFTUVOVCwgTUVSQ0hBTlRBQklMSVRZLCBvciBGSVRORVNTIEZPUiBBCiAgICAgIFBBUlRJQ1VMQVIgUFVSUE9TRS4gWW91IGFyZSBzb2xlbHkgcmVzcG9uc2libGUgZm9yIGRldGVybWluaW5nIHRoZQogICAgICBhcHByb3ByaWF0ZW5lc3Mgb2YgdXNpbmcgb3IgcmVkaXN0cmlidXRpbmcgdGhlIFdvcmsgYW5kIGFzc3VtZSBhbnkKICAgICAgcmlza3MgYXNzb2NpYXRlZCB3aXRoIFlvdXIgZXhlcmNpc2Ugb2YgcGVybWlzc2lvbnMgdW5kZXIgdGhpcyBMaWNlbnNlLgoKICAgOC4gTGltaXRhdGlvbiBvZiBMaWFiaWxpdHkuIEluIG5vIGV2ZW50IGFuZCB1bmRlciBubyBsZWdhbCB0aGVvcnksCiAgICAgIHdoZXRoZXIgaW4gdG9ydCAoaW5jbHVkaW5nIG5lZ2xpZ2VuY2UpLCBjb250cmFjdCwgb3Igb3RoZXJ3aXNlLAogICAgICB1bmxlc3MgcmVxdWlyZWQgYnkgYXBwbGljYWJsZSBsYXcgKHN1Y2ggYXMgZGVsaWJlcmF0ZSBhbmQgZ3Jvc3NseQogICAgICBuZWdsaWdlbnQgYWN0cykgb3IgYWdyZWVkIHRvIGluIHdyaXRpbmcsIHNoYWxsIGFueSBDb250cmlidXRvciBiZQogICAgICBsaWFibGUgdG8gWW91IGZvciBkYW1hZ2VzLCBpbmNsdWRpbmcgYW55IGRpcmVjdCwgaW5kaXJlY3QsIHNwZWNpYWwsCiAgICAgIGluY2lkZW50YWwsIG9yIGNvbnNlcXVlbnRpYWwgZGFtYWdlcyBvZiBhbnkgY2hhcmFjdGVyIGFyaXNpbmcgYXMgYQogICAgICByZXN1bHQgb2YgdGhpcyBMaWNlbnNlIG9yIG91dCBvZiB0aGUgdXNlIG9yIGluYWJpbGl0eSB0byB1c2UgdGhlCiAgICAgIFdvcmsgKGluY2x1ZGluZyBidXQgbm90IGxpbWl0ZWQgdG8gZGFtYWdlcyBmb3IgbG9zcyBvZiBnb29kd2lsbCwKICAgICAgd29yayBzdG9wcGFnZSwgY29tcHV0ZXIgZmFpbHVyZSBvciBtYWxmdW5jdGlvbiwgb3IgYW55IGFuZCBhbGwKICAgICAgb3RoZXIgY29tbWVyY2lhbCBkYW1hZ2VzIG9yIGxvc3NlcyksIGV2ZW4gaWYgc3VjaCBDb250cmlidXRvcgogICAgICBoYXMgYmVlbiBhZHZpc2VkIG9mIHRoZSBwb3NzaWJpbGl0eSBvZiBzdWNoIGRhbWFnZXMuCgogICA5LiBBY2NlcHRpbmcgV2FycmFudHkgb3IgQWRkaXRpb25hbCBMaWFiaWxpdHkuIFdoaWxlIHJlZGlzdHJpYnV0aW5nCiAgICAgIHRoZSBXb3JrIG9yIERlcml2YXRpdmUgV29ya3MgdGhlcmVvZiwgWW91IG1heSBjaG9vc2UgdG8gb2ZmZXIsCiAgICAgIGFuZCBjaGFyZ2UgYSBmZWUgZm9yLCBhY2NlcHRhbmNlIG9mIHN1cHBvcnQsIHdhcnJhbnR5LCBpbmRlbW5pdHksCiAgICAgIG9yIG90aGVyIGxpYWJpbGl0eSBvYmxpZ2F0aW9ucyBhbmQvb3IgcmlnaHRzIGNvbnNpc3RlbnQgd2l0aCB0aGlzCiAgICAgIExpY2Vuc2UuIEhvd2V2ZXIsIGluIGFjY2VwdGluZyBzdWNoIG9ibGlnYXRpb25zLCBZb3UgbWF5IGFjdCBvbmx5CiAgICAgIG9uIFlvdXIgb3duIGJlaGFsZiBhbmQgb24gWW91ciBzb2xlIHJlc3BvbnNpYmlsaXR5LCBub3Qgb24gYmVoYWxmCiAgICAgIG9mIGFueSBvdGhlciBDb250cmlidXRvciwgYW5kIG9ubHkgaWYgWW91IGFncmVlIHRvIGluZGVtbmlmeSwKICAgICAgZGVmZW5kLCBhbmQgaG9sZCBlYWNoIENvbnRyaWJ1dG9yIGhhcm1sZXNzIGZvciBhbnkgbGlhYmlsaXR5CiAgICAgIGluY3VycmVkIGJ5LCBvciBjbGFpbXMgYXNzZXJ0ZWQgYWdhaW5zdCwgc3VjaCBDb250cmlidXRvciBieSByZWFzb24KICAgICAgb2YgeW91ciBhY2NlcHRpbmcgYW55IHN1Y2ggd2FycmFudHkgb3IgYWRkaXRpb25hbCBsaWFiaWxpdHkuCgogICBFTkQgT0YgVEVSTVMgQU5EIENPTkRJVElPTlMKCiAgIEFQUEVORElYOiBIb3cgdG8gYXBwbHkgdGhlIEFwYWNoZSBMaWNlbnNlIHRvIHlvdXIgd29yay4KCiAgICAgIFRvIGFwcGx5IHRoZSBBcGFjaGUgTGljZW5zZSB0byB5b3VyIHdvcmssIGF0dGFjaCB0aGUgZm9sbG93aW5nCiAgICAgIGJvaWxlcnBsYXRlIG5vdGljZSwgd2l0aCB0aGUgZmllbGRzIGVuY2xvc2VkIGJ5IGJyYWNrZXRzICJbXSIKICAgICAgcmVwbGFjZWQgd2l0aCB5b3VyIG93biBpZGVudGlmeWluZyBpbmZvcm1hdGlvbi4gKERvbid0IGluY2x1ZGUKICAgICAgdGhlIGJyYWNrZXRzISkgIFRoZSB0ZXh0IHNob3VsZCBiZSBlbmNsb3NlZCBpbiB0aGUgYXBwcm9wcmlhdGUKICAgICAgY29tbWVudCBzeW50YXggZm9yIHRoZSBmaWxlIGZvcm1hdC4gV2UgYWxzbyByZWNvbW1lbmQgdGhhdCBhCiAgICAgIGZpbGUgb3IgY2xhc3MgbmFtZSBhbmQgZGVzY3JpcHRpb24gb2YgcHVycG9zZSBiZSBpbmNsdWRlZCBvbiB0aGUKICAgICAgc2FtZSAicHJpbnRlZCBwYWdlIiBhcyB0aGUgY29weXJpZ2h0IG5vdGljZSBmb3IgZWFzaWVyCiAgICAgIGlkZW50aWZpY2F0aW9uIHdpdGhpbiB0aGlyZC1wYXJ0eSBhcmNoaXZlcy4KCiAgIENvcHlyaWdodCBbeXl5eV0gW25hbWUgb2YgY29weXJpZ2h0IG93bmVyXQoKICAgTGljZW5zZWQgdW5kZXIgdGhlIEFwYWNoZSBMaWNlbnNlLCBWZXJzaW9uIDIuMCAodGhlICJMaWNlbnNlIik7CiAgIHlvdSBtYXkgbm90IHVzZSB0aGlzIGZpbGUgZXhjZXB0IGluIGNvbXBsaWFuY2Ugd2l0aCB0aGUgTGljZW5zZS4KICAgWW91IG1heSBvYnRhaW4gYSBjb3B5IG9mIHRoZSBMaWNlbnNlIGF0CgogICAgICAgaHR0cDovL3d3dy5hcGFjaGUub3JnL2xpY2Vuc2VzL0xJQ0VOU0UtMi4wCgogICBVbmxlc3MgcmVxdWlyZWQgYnkgYXBwbGljYWJsZSBsYXcgb3IgYWdyZWVkIHRvIGluIHdyaXRpbmcsIHNvZnR3YXJlCiAgIGRpc3RyaWJ1dGVkIHVuZGVyIHRoZSBMaWNlbnNlIGlzIGRpc3RyaWJ1dGVkIG9uIGFuICJBUyBJUyIgQkFTSVMsCiAgIFdJVEhPVVQgV0FSUkFOVElFUyBPUiBDT05ESVRJT05TIE9GIEFOWSBLSU5ELCBlaXRoZXIgZXhwcmVzcyBvciBpbXBsaWVkLgogICBTZWUgdGhlIExpY2Vuc2UgZm9yIHRoZSBzcGVjaWZpYyBsYW5ndWFnZSBnb3Zlcm5pbmcgcGVybWlzc2lvbnMgYW5kCiAgIGxpbWl0YXRpb25zIHVuZGVyIHRoZSBMaWNlbnNlLgo=")
EXPECTED_ENGINE_SHA256 = "bc8a54879ef02c7ea64b8b333d6a976f0ea65c4949149d01f463f23bccee653e"
EXPECTED_SCHEMA_SHA256 = "a82c89c1a2315b93f39775d8e025471a01b738647c9772658368ee6b1b6f4867"
EXPECTED_ENGINE_LICENSE_SHA256 = "c71d239df91726fc519c6eb72d318ec65820627232b2f796219e87dcf35d0ab4"
EXPECTED_ACTIONS_SHA256 = ['e95796bd0054b6bd83405e1ac7c7851c57bd32ca971e6d60f47126d1fac9bcd8', '30d7dc0e8aacad13c6ab65510539d13e0183066c0988523dc5083b4599ffc44e']
EXPECTED_REWARDS = [87467.0, 87459.0]

def sha256(data):
    return hashlib.sha256(data).hexdigest()

assert sha256(ENGINE_SOURCE) == EXPECTED_ENGINE_SHA256
assert sha256(ENGINE_SCHEMA) == EXPECTED_SCHEMA_SHA256
assert sha256(ENGINE_LICENSE) == EXPECTED_ENGINE_LICENSE_SHA256
try:
    HOST_CORE_VERSION = importlib.metadata.version("kaggle-environments")
except importlib.metadata.PackageNotFoundError:
    HOST_CORE_VERSION = "distribution metadata unavailable"
print("Observed Kaggle host core before engine import:", HOST_CORE_VERSION)
SEED_HELPER_SHIM_INSTALLED = False
PINNED_GAME_DIR = Path(RUNTIME_CLEANUP.name) / "kaggriculture-engine"
PINNED_GAME_DIR.mkdir(parents=True, exist_ok=True)
ENGINE_FILE = PINNED_GAME_DIR / "kaggriculture.py"
SCHEMA_FILE = PINNED_GAME_DIR / "kaggriculture.json"
LICENSE_FILE = PINNED_GAME_DIR / "LICENSE"
ENGINE_FILE.write_bytes(ENGINE_SOURCE)
SCHEMA_FILE.write_bytes(ENGINE_SCHEMA)
LICENSE_FILE.write_bytes(ENGINE_LICENSE)

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    saved_out, saved_err = os.dup(1), os.dup(2)
    sink = os.open(os.devnull, os.O_WRONLY)
    try:
        os.dup2(sink, 1)
        os.dup2(sink, 2)
        from kaggle_environments import utils as _kaggle_utils
        if not hasattr(_kaggle_utils, "resolve_episode_seed"):
            # Copyright 2020 Kaggle Inc.; Apache-2.0. Exact source from kaggle_environments/utils.py.
            from typing import Any, Callable
            import random
            _VENDORED_SEED_HELPER_SOURCE = 'def resolve_episode_seed(\n    env: Any,\n    *,\n    config_key: str = "seed",\n    fallback: Callable[[], int] | None = None,\n) -> int:\n    """Resolve, scrub, and persist an episode seed for envs with hidden state.\n\n    Used by interpreters whose initial state depends on a seed that agents\n    must not be able to read (e.g. random maze layout, comet schedules,\n    weather rolls). The seed is taken from the first available source:\n    ``env.info["seed"]`` (preserved across re-initialization), then\n    ``configuration[config_key]``, then ``fallback()`` (defaulting to a random\n    31-bit int). The value is then cleared from ``configuration`` so agents\n    can\'t read it via the observation, and stored on ``env.info["seed"]`` so\n    it persists into the replay.\n    """\n    if not hasattr(env, "info") or env.info is None:\n        env.info = {}\n    seed = env.info.get("seed")\n    config = env.configuration\n    if seed is None:\n        seed = getattr(config, config_key, None)\n        if seed is None and isinstance(config, dict):\n            seed = config.get(config_key)\n    if seed is None:\n        seed = fallback() if fallback is not None else random.randrange(2**31)\n    try:\n        setattr(config, config_key, None)\n    except (AttributeError, TypeError):\n        config[config_key] = None\n    env.info["seed"] = seed\n    return seed'
            EXPECTED_SEED_HELPER_SHA256 = "15feb82a21849d8bb07d0bad806dc287992af24bc8d0370031f40c4b1b856a54"
            assert sha256(_VENDORED_SEED_HELPER_SOURCE.encode("utf-8")) == EXPECTED_SEED_HELPER_SHA256
            _seed_helper_namespace = {"Any": Any, "Callable": Callable, "random": random}
            exec(compile(_VENDORED_SEED_HELPER_SOURCE, "kaggle-utils-resolve-episode-seed", "exec"), _seed_helper_namespace)
            _kaggle_utils.resolve_episode_seed = _seed_helper_namespace["resolve_episode_seed"]
            SEED_HELPER_SHIM_INSTALLED = True
        engine_spec = importlib.util.spec_from_file_location("kaggriculture_pinned_1327", ENGINE_FILE)
        assert engine_spec is not None and engine_spec.loader is not None
        pinned_engine = importlib.util.module_from_spec(engine_spec)
        engine_spec.loader.exec_module(pinned_engine)
        from kaggle_environments import environments, make, register
        assert callable(make) and callable(register), "Installed Kaggle core lacks make/register APIs."
        register("kaggriculture", {
            "agents": pinned_engine.agents,
            "html_renderer": pinned_engine.html_renderer,
            "interpreter": pinned_engine.interpreter,
            "renderer": pinned_engine.renderer,
            "specification": pinned_engine.specification,
        })
    finally:
        os.dup2(saved_out, 1)
        os.dup2(saved_err, 2)
        os.close(saved_out)
        os.close(saved_err)
        os.close(sink)
registered = environments["kaggriculture"]
assert registered["interpreter"] is pinned_engine.interpreter
assert registered["specification"] == json.loads(ENGINE_SCHEMA)
if SEED_HELPER_SHIM_INSTALLED:
    print("Installed byte-verified Kaggle seed-helper compatibility shim:", EXPECTED_SEED_HELPER_SHA256)
print("Registered exact Kaggriculture engine SHA-256:", EXPECTED_ENGINE_SHA256)
print("Registered exact schema SHA-256:", EXPECTED_SCHEMA_SHA256)

In [4]:
import contextlib, hashlib, io, json, os, sys
from contextlib import contextmanager

@contextmanager
def silence_native_output():
    sys.stdout.flush()
    sys.stderr.flush()
    saved_out, saved_err = os.dup(1), os.dup(2)
    sink = os.open(os.devnull, os.O_WRONLY)
    try:
        os.dup2(sink, 1)
        os.dup2(sink, 2)
        yield
    finally:
        os.dup2(saved_out, 1)
        os.dup2(saved_err, 2)
        os.close(saved_out)
        os.close(saved_err)
        os.close(sink)

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()), silence_native_output():
    env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": 861795495})
    steps = env.run([str(MAIN), str(CONTROL)])
assert len(steps) == 720
final = steps[-1]
statuses = [agent["status"] for agent in final]
rewards = [agent["reward"] for agent in final]
callbacks_per_agent = [sum(state[i].get("action") is not None for state in steps[1:]) for i in range(2)]
actions = [[row[i].get("action") for row in steps[1:]] for i in range(2)]
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
actions_sha256 = [digest([digest(action) for action in seat]) for seat in actions]
assert statuses == ["DONE", "DONE"]
assert callbacks_per_agent == [719, 719]
assert all(isinstance(value, (int, float)) for value in rewards)
assert actions_sha256 == EXPECTED_ACTIONS_SHA256, "Pinned full-game action stream mismatch."
assert rewards == EXPECTED_REWARDS, "Pinned full-game rewards mismatch."
print("PASS: host core", HOST_CORE_VERSION, "used the exact pinned Kaggriculture source/schema.")
print("PASS: 720 states; DONE/DONE; 719 callbacks per seat.")
print("PASS: action SHA-256", actions_sha256, "rewards", rewards)
RUNTIME_CLEANUP.cleanup()
assert not Path(str(MAIN)).exists() and not Path(str(NOTICE)).exists()
print("PASS: temporary policy, game source, schema and license removed; submission archive remains.")